# Fine-tune the two models (free Colab GPU)

**You do not need to upload or zip anything.** This notebook already contains all the project code and creates the files itself.

**Before you start:** menu *Runtime -> Change runtime type -> **T4 GPU** -> Save*. Then run the cells below **one by one, in order** (click the play button on each, wait for it to finish).

### 1. Check that a GPU is attached (you should see a table that mentions a Tesla T4)

In [ ]:
!nvidia-smi

### 2. Create the project files inside Colab (just run it - there is no need to read the code)

In [ ]:
%%bash
set -e
mkdir -p sample_docs
cat > retriever.py << '__END_OF_FILE__'
"""RAG retriever: paragraph-aware chunking + TF-IDF cosine-similarity search.

Dependency-light on purpose (no embedding API, no model download), so it runs
offline. To upgrade to neural embeddings later, keep the same three methods
(`index`, `retrieve_scored`, `best_support`) in a new class and swap it in.
"""
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import List, Optional, Tuple

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass
class Chunk:
    text: str
    source: str
    chunk_id: str = ""


def _windows(words: List[str], max_words: int, overlap: int):
    step = max(max_words - overlap, 1)
    for start in range(0, len(words), step):
        yield words[start:start + max_words]
        if start + max_words >= len(words):
            break


def chunk_text(text: str, source: str, max_words: int = 120, overlap: int = 20) -> List[Chunk]:
    """Split on blank lines, merge short paragraphs up to `max_words`, and only
    fall back to overlapping word windows for a single over-long paragraph.
    Chunks therefore end on paragraph boundaries instead of mid-sentence."""
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    pieces: List[str] = []
    buf: List[str] = []

    def flush() -> None:
        nonlocal buf
        if buf:
            pieces.append(" ".join(buf))
            buf = []

    for para in paragraphs:
        words = para.split()
        if len(words) > max_words:
            flush()
            pieces.extend(" ".join(w) for w in _windows(words, max_words, overlap))
        elif len(buf) + len(words) > max_words:
            flush()
            buf = list(words)
        else:
            buf.extend(words)
    flush()
    return [Chunk(text=p, source=source, chunk_id=f"{source}#{i + 1}") for i, p in enumerate(pieces)]


class TfidfRetriever:
    def __init__(self) -> None:
        self._chunks: List[Chunk] = []
        self._vectorizer: Optional[TfidfVectorizer] = None
        self._matrix = None

    def index(self, chunks: List[Chunk]) -> None:
        if not chunks:
            raise ValueError("No chunks to index - the document folder is empty.")
        self._chunks = chunks
        self._vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
        self._matrix = self._vectorizer.fit_transform([c.text for c in chunks])

    @property
    def chunks(self) -> List[Chunk]:
        return list(self._chunks)

    def retrieve_scored(self, query: str, top_k: int = 4, min_score: float = 0.0) -> List[Tuple[Chunk, float]]:
        if self._vectorizer is None:
            raise RuntimeError("Call index() before retrieving.")
        scores = cosine_similarity(self._vectorizer.transform([query]), self._matrix)[0]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [(self._chunks[i], float(scores[i])) for i in ranked[:top_k] if scores[i] > min_score]

    def retrieve(self, query: str, top_k: int = 4) -> List[Chunk]:
        return [c for c, _ in self.retrieve_scored(query, top_k)]

    def best_support(self, text: str, chunks: List[Chunk]) -> Tuple[Optional[Chunk], float]:
        """How well is `text` supported by this specific set of chunks?
        Returns the closest chunk and its cosine similarity (0 when nothing overlaps)."""
        if self._vectorizer is None:
            raise RuntimeError("Call index() before scoring.")
        if not chunks:
            return None, 0.0
        vecs = self._vectorizer.transform([text] + [c.text for c in chunks])
        sims = cosine_similarity(vecs[0], vecs[1:])[0]
        best = int(sims.argmax())
        return chunks[best], float(sims[best])
__END_OF_FILE__
cat > llm_clients.py << '__END_OF_FILE__'
"""LLM clients.

Default setup = two *different* local Ollama models, one per LLM step:
  LLM 1 (requirements)  -> fine-tuned `blockchain-req-llm`  if installed, else base `llama3.2:3b`
  LLM 2 (SDLC only)     -> fine-tuned `blockchain-sdlc-llm` if installed, else base `qwen2.5:3b`
Gemini is available as an optional cloud alternative.
"""
from __future__ import annotations

import json
import os
import time
import urllib.error
import urllib.request
from abc import ABC, abstractmethod
from typing import List, Optional

# role -> (fine-tuned model name created by install_models.py, base model fallback)
DEFAULT_OLLAMA_MODELS = {
    "llm1": ("blockchain-req-llm", "llama3.2:3b"),
    "llm2": ("blockchain-sdlc-llm", "qwen2.5:3b"),
}
DEFAULT_GEMINI_MODEL = "gemini-3.5-flash-lite"
DEFAULT_OLLAMA_HOST = "http://localhost:11434"


class LLMError(RuntimeError):
    """Raised for any model/connection problem, with a human-readable message."""


def extract_json(text: str) -> dict:
    """Parse a JSON object out of model output, tolerating ``` fences and chatter."""
    text = (text or "").strip()
    if text.startswith("```"):
        text = text.split("```", 2)[1]
        if text.lower().startswith("json"):
            text = text[4:]
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object found in model output")
    return json.loads(text[start:end + 1])


class LLMClient(ABC):
    label = "llm"

    @abstractmethod
    def _complete(self, system: str, user: str) -> str:
        """Return the raw text reply from the model."""

    def generate_json(self, system: str, user: str, retries: int = 1) -> dict:
        """Call the model and return parsed JSON; re-ask once if the reply wasn't valid JSON."""
        prompt, last_err = user, None
        for _ in range(retries + 1):
            raw = self._complete(system, prompt)
            try:
                return extract_json(raw)
            except ValueError as e:  # json.JSONDecodeError is a ValueError too
                last_err = e
                prompt = user + "\n\nYour previous reply was not valid JSON. Reply with ONLY the JSON object, nothing else."
        raise LLMError(f"{self.label} did not return valid JSON ({last_err}).")


class OllamaClient(LLMClient):
    """Local model served by Ollama (https://ollama.com). Free, offline, no API key."""

    def __init__(self, model: str, host: Optional[str] = None, timeout: int = 900, num_ctx: int = 4096):
        self.model = model
        self.host = (host or os.environ.get("OLLAMA_HOST") or DEFAULT_OLLAMA_HOST).rstrip("/")
        if not self.host.startswith("http"):
            self.host = "http://" + self.host
        self.timeout, self.num_ctx = timeout, num_ctx
        self.label = f"Ollama '{model}'"

    def _complete(self, system: str, user: str) -> str:
        payload = json.dumps({
            "model": self.model,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "format": "json",
            "stream": False,
            "options": {"temperature": 0.2, "num_ctx": self.num_ctx, "num_predict": 1500},
        }).encode("utf-8")
        req = urllib.request.Request(f"{self.host}/api/chat", data=payload, headers={"Content-Type": "application/json"})
        print(f"  -> {self.label} is generating (this can take a minute on CPU)...", flush=True)
        try:
            with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))["message"]["content"]
        except urllib.error.HTTPError as e:
            body = e.read().decode("utf-8", "ignore")
            if e.code == 404:
                raise LLMError(
                    f"Ollama has no model named '{self.model}'. Install it with `ollama pull {self.model}` "
                    f"(base models) or `python install_models.py` (your fine-tuned models). Installed: "
                    f"{', '.join(ollama_installed_models(self.host) or []) or 'none'}."
                ) from e
            raise LLMError(f"Ollama error {e.code}: {body[:300]}") from e
        except (urllib.error.URLError, ConnectionError, TimeoutError) as e:
            raise LLMError(
                f"Cannot reach Ollama at {self.host}. Start it (open the Ollama app, or run `ollama serve`) "
                f"and try again. Details: {e}"
            ) from e


class GeminiClient(LLMClient):
    """Optional cloud model. Needs `pip install google-genai` and a GEMINI_API_KEY."""

    def __init__(self, model: str = DEFAULT_GEMINI_MODEL, api_key: Optional[str] = None):
        key = api_key or os.environ.get("GEMINI_API_KEY")
        if not key:
            raise LLMError("GEMINI_API_KEY is missing. Set the environment variable or paste the key in the web page.")
        try:
            from google import genai
        except ImportError as e:
            raise LLMError("The Gemini provider needs `pip install google-genai`.") from e
        self.model = model
        self.client = genai.Client(api_key=key)
        self.label = f"Gemini '{model}'"

    def _complete(self, system: str, user: str) -> str:
        from google.genai import types
        last_err = None
        for attempt in range(1, 4):
            try:
                print(f"  -> {self.label} (attempt {attempt})...", flush=True)
                resp = self.client.models.generate_content(
                    model=self.model,
                    contents=user,
                    config=types.GenerateContentConfig(
                        system_instruction=system, temperature=0.2, response_mime_type="application/json"),
                )
                return resp.text
            except Exception as e:  # retry only transient problems, fail fast on bad keys / bad model names
                last_err = e
                if not any(t in str(e) for t in ("429", "500", "503", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "overloaded")):
                    break
                time.sleep(3 * attempt)
        raise LLMError(f"{self.label} failed: {last_err}")


def ollama_installed_models(host: Optional[str] = None) -> Optional[List[str]]:
    """Names of models installed in Ollama, or None if the Ollama server is not reachable."""
    host = (host or os.environ.get("OLLAMA_HOST") or DEFAULT_OLLAMA_HOST).rstrip("/")
    if not host.startswith("http"):
        host = "http://" + host
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=3) as resp:
            return [m["name"] for m in json.loads(resp.read().decode("utf-8")).get("models", [])]
    except Exception:
        return None


def pick_default_ollama_model(role: str, installed: Optional[List[str]]) -> dict:
    """Prefer the fine-tuned model for this role when it is installed, else the base model."""
    finetuned, base = DEFAULT_OLLAMA_MODELS[role]
    names = {n.split(":")[0] for n in (installed or [])}
    if finetuned in names:
        return {"model": finetuned, "finetuned": True}
    return {"model": base, "finetuned": False}


def build_client(provider: str, model: Optional[str] = None, api_key: Optional[str] = None, role: str = "llm1") -> LLMClient:
    provider = (provider or "ollama").lower()
    if provider == "ollama":
        return OllamaClient(model or pick_default_ollama_model(role, ollama_installed_models())["model"])
    if provider == "gemini":
        return GeminiClient(model or DEFAULT_GEMINI_MODEL, api_key=api_key)
    raise LLMError(f"Unknown provider {provider!r}. Choose 'ollama' or 'gemini'.")
__END_OF_FILE__
cat > pipeline.py << '__END_OF_FILE__'
"""Pipeline (retrieve first, then generate - the order used by retrieval-grounded generation):

  input (domain + functionality)
    -> Step 1  RAG retrieval: find the most relevant chunks of the existing requirement documents
    -> Step 2  LLM 1: generate requirements + reasoning, grounded in that evidence
               (+ a deterministic check of how well each requirement is supported by the evidence)
    -> Step 3  LLM 2: identify the SDLC model (its ONLY job) from the requirements + evidence
    -> final output

The prompt builders below are also used by generate_dataset.py, so the fine-tuning data
uses exactly the same prompts the pipeline uses at run time.
"""
from __future__ import annotations

import os
from typing import Callable, Dict, List, Optional, Tuple

from llm_clients import LLMClient, LLMError
from retriever import Chunk, TfidfRetriever, chunk_text

TEXT_EXTENSIONS = (".txt", ".md")
MIN_EVIDENCE_SCORE = 0.02      # chunks scoring below this are treated as irrelevant
MIN_GROUNDING = 0.10           # requirement/evidence similarity below this => flag for human review

SDLC_LABELS = [
    "Waterfall", "V-Model", "Spiral", "Agile", "DevSecOps",
    "Agile-V-Model Hybrid", "DevSecOps-V-Model Hybrid",
]

SDLC_HEURISTICS = """Reference SDLC heuristics:
- Stable requirements and extensive approvals needed -> Waterfall
- Strict verification and validation needed (e.g. formally verified logic) -> V-Model
- High uncertainty or technical risk -> Spiral
- Frequently changing requirements -> Agile
- Continuous secure deployment with rigorous automated security testing -> DevSecOps
- High regulation with evolving requirements and high-stakes deployment -> Agile-V-Model or DevSecOps hybrid"""

STAGE1_SCHEMA = """Return ONLY strict JSON (no markdown fences):
{
 "requirements": [{"id": "REQ-001", "statement": "one clear, testable requirement", "category": "short category"}],
 "reasoning": "one paragraph explaining how the evidence and the functionality led to these requirements"
}"""

STAGE2_SCHEMA = """Return ONLY strict JSON (no markdown fences). Identifying the SDLC model is your ONLY task:
{
 "recommended_sdlc": "model name",
 "key_factors": ["3-4 short project characteristics, drawn from the requirements and evidence, that drove this choice"],
 "justification": "a software-engineering justification paragraph, briefly noting why the main alternative(s) fit less well"
}"""


# ----------------------------------------------------------------------------- documents / index
def load_documents(folder: str, extra_text: str = "") -> List[Tuple[str, str]]:
    """Read every .txt/.md file in `folder`; optionally add pasted text as one more document."""
    docs: List[Tuple[str, str]] = []
    if folder:
        if not os.path.isdir(folder):
            raise ValueError(f"Document folder not found: {folder}")
        for name in sorted(os.listdir(folder)):
            path = os.path.join(folder, name)
            if os.path.isfile(path) and name.lower().endswith(TEXT_EXTENSIONS):
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    docs.append((name, f.read()))
    if extra_text and extra_text.strip():
        docs.append(("pasted_text", extra_text))
    if not docs:
        raise ValueError("No requirement documents found (.txt/.md files, or pasted text).")
    return docs


def build_index(documents: List[Tuple[str, str]]) -> TfidfRetriever:
    chunks: List[Chunk] = []
    for source, text in documents:
        chunks.extend(chunk_text(text, source))
    retriever = TfidfRetriever()
    retriever.index(chunks)
    return retriever


# ----------------------------------------------------------------------------- prompts
def format_evidence(evidence: List[dict]) -> str:
    if not evidence:
        return "(no relevant evidence was found in the existing documents)"
    return "\n\n".join(f"[{e['id']}] (source: {e['source']}) {e['text']}" for e in evidence)


def stage1_messages(domain: str, functionality: str, evidence: List[dict]) -> Tuple[str, str]:
    system = f"You are a requirements engineer for the '{domain}' domain in blockchain-based finance."
    user = (
        f'Functionality: "{functionality}"\n\n'
        f'Evidence retrieved from the existing requirement documents:\n"""\n{format_evidence(evidence)}\n"""\n\n'
        "Write 4-6 specific, testable requirements for this functionality. Base them on the evidence wherever it "
        "applies; add standard domain requirements only where the evidence is silent. Each statement must be one "
        "sentence using \"shall\". Then explain your reasoning in one paragraph.\n\n" + STAGE1_SCHEMA
    )
    return system, user


def requirements_text(requirements: List[dict]) -> str:
    return "\n".join(f"{r.get('id', '')}: {r.get('statement', '')} [{r.get('category', '')}]" for r in requirements)


def stage2_messages(domain: str, requirements: List[dict], evidence: List[dict]) -> Tuple[str, str]:
    system = (
        f"You identify the single most appropriate SDLC model for a blockchain-finance project in the '{domain}' "
        "domain. This is your only task -- do not propose or revise requirements."
    )
    user = (
        f'Requirements:\n"""\n{requirements_text(requirements)}\n"""\n\n'
        f'Supporting evidence from the existing documents:\n"""\n{format_evidence(evidence)}\n"""\n\n'
        f"{SDLC_HEURISTICS}\n\n"
        "Using the requirements and the evidence together, identify the single most appropriate SDLC model "
        f"(choose from: {', '.join(SDLC_LABELS)}).\n\n" + STAGE2_SCHEMA
    )
    return system, user


# ----------------------------------------------------------------------------- normalisation (small models are messy)
def _get(d: dict, *names: str, default=""):
    lowered = {str(k).lower(): v for k, v in d.items()}
    for n in names:
        if n in lowered and lowered[n] not in (None, ""):
            return lowered[n]
    return default


def normalize_requirements(raw) -> List[dict]:
    items = raw if isinstance(raw, list) else []
    out: List[dict] = []
    for item in items:
        if isinstance(item, str):
            item = {"statement": item}
        if not isinstance(item, dict):
            continue
        statement = str(_get(item, "statement", "requirement", "text", "description")).strip()
        if not statement:
            continue
        out.append({
            "id": str(_get(item, "id", default="")).strip() or f"REQ-{len(out) + 1:03d}",
            "statement": statement,
            "category": str(_get(item, "category", "type", default="General")).strip() or "General",
        })
    return out


def normalize_sdlc(raw: dict) -> dict:
    factors = _get(raw, "key_factors", "factors", default=[])
    if isinstance(factors, str):
        factors = [factors]
    return {
        "recommended_sdlc": str(_get(raw, "recommended_sdlc", "sdlc", "model", default="")).strip(),
        "key_factors": [str(f) for f in factors if str(f).strip()],
        "justification": str(_get(raw, "justification", "reasoning", default="")).strip(),
    }


# ----------------------------------------------------------------------------- the three steps
def retrieve_evidence(retriever: TfidfRetriever, domain: str, functionality: str, top_k: int = 4) -> List[dict]:
    """Step 1 - RAG: query the existing documents with the user's input."""
    hits = retriever.retrieve_scored(f"{domain}: {functionality}", top_k=top_k, min_score=MIN_EVIDENCE_SCORE)
    return [{"id": f"E{i + 1}", "source": c.source, "text": c.text, "score": round(s, 3)} for i, (c, s) in enumerate(hits)]


def generate_requirements(domain: str, functionality: str, evidence: List[dict], llm1: LLMClient) -> dict:
    """Step 2 - LLM 1 drafts requirements, grounded in the retrieved evidence."""
    system, user = stage1_messages(domain, functionality, evidence)
    raw = llm1.generate_json(system, user)
    requirements = normalize_requirements(_get(raw, "requirements", default=[]))
    if not requirements:
        raise LLMError(f"{llm1.label} returned no usable requirements. Try again or use a larger model.")
    return {"requirements": requirements, "reasoning": str(_get(raw, "reasoning", "explanation", default="")).strip()}


def ground_requirements(retriever: TfidfRetriever, requirements: List[dict], evidence: List[dict],
                        min_grounding: float = MIN_GROUNDING) -> List[dict]:
    """Deterministic check (no LLM): how closely does each requirement match the retrieved evidence?
    Requirements with weak support are flagged for human review."""
    chunks = [Chunk(text=e["text"], source=e["source"]) for e in evidence]
    out = []
    for r in requirements:
        chunk, score = retriever.best_support(r["statement"], chunks)
        out.append({**r,
                    "support_source": chunk.source if chunk else None,
                    "support_score": round(score, 3),
                    "needs_review": score < min_grounding})
    return out


def identify_sdlc(domain: str, requirements: List[dict], evidence: List[dict], llm2: LLMClient) -> dict:
    """Step 3 - LLM 2: its only job is naming the SDLC model, with a justification."""
    system, user = stage2_messages(domain, requirements, evidence)
    sdlc = normalize_sdlc(llm2.generate_json(system, user))
    if not sdlc["recommended_sdlc"]:
        raise LLMError(f"{llm2.label} did not name an SDLC model. Try again or use a larger model.")
    return sdlc


def run_pipeline(retriever: TfidfRetriever, domain: str, functionality: str, llm1: LLMClient, llm2: LLMClient,
                 top_k: int = 4, min_grounding: float = MIN_GROUNDING,
                 progress: Optional[Callable[[str], None]] = None) -> Dict:
    say = progress or (lambda _msg: None)
    say("Step 1/3: retrieving evidence (RAG)")
    evidence = retrieve_evidence(retriever, domain, functionality, top_k)
    say("Step 2/3: LLM 1 generating requirements")
    draft = generate_requirements(domain, functionality, evidence, llm1)
    requirements = ground_requirements(retriever, draft["requirements"], evidence, min_grounding)
    say("Step 3/3: LLM 2 identifying the SDLC model")
    sdlc = identify_sdlc(domain, requirements, evidence, llm2)
    return {
        "input": {"domain": domain, "functionality": functionality},
        "evidence": evidence,
        "requirements": requirements,
        "requirements_reasoning": draft["reasoning"],
        "needs_review_count": sum(1 for r in requirements if r["needs_review"]),
        "sdlc": sdlc,
        "models": {"llm1": llm1.label, "llm2": llm2.label},
    }
__END_OF_FILE__
cat > seed_data.py << '__END_OF_FILE__'
"""Hand-authored seed data for fine-tuning (28 examples across 7 blockchain-finance domains).

Each seed is ONE project scenario and feeds BOTH fine-tuning sets:
  * Stage 1 (LLM 1): domain + functionality + retrieved evidence  ->  requirements + reasoning
  * Stage 2 (LLM 2): the same requirements + evidence             ->  SDLC model + factors + justification

The SDLC labels are deliberately varied (not always "hybrid"), following the heuristics in pipeline.py.
These are illustrative training examples written for this project - review and edit them (and add your own,
ideally from real documents your course/organisation provides) before relying on the fine-tuned models.
"""
from __future__ import annotations


def S(domain, functionality, reqs, reasoning, sdlc, factors, justification):
    return {"domain": domain, "functionality": functionality, "requirements": reqs,
            "reasoning": reasoning, "sdlc": sdlc, "factors": factors, "justification": justification}


DEFI, XB, KYC, STB, CUS, TF, CBDC = (
    "DeFi Lending", "Cross-Border Payments", "Blockchain KYC / AML", "Stablecoin Issuance",
    "Crypto Custody", "Tokenized Trade Finance", "CBDC")

SEEDS = [
    # ------------------------------------------------------------------ DeFi Lending
    S(DEFI, "Multi-signature approval for large withdrawals from a lending pool",
      [("Functional", "The pool contract shall execute withdrawals below 10,000 USDC equivalent immediately once collateral health checks pass."),
       ("Security", "Withdrawals above 10,000 USDC equivalent shall enter an approval queue and execute only after at least 3 of 5 authorized signers approve."),
       ("Security", "Queued withdrawals shall be subject to a 24-hour timelock during which authorized monitors can cancel a transfer flagged as anomalous."),
       ("Auditability", "Every approval, cancellation and execution shall emit an on-chain event recording the requester, amount, signers and timestamp.")],
      "The policy notes record incidents where one compromised key authorized a large transfer, while normal users still need instant withdrawals. A value threshold keeps routine flows fast, and multi-party approval, a timelock and immutable event logs remove the single point of failure and give auditors a verifiable trail.",
      "DevSecOps-V-Model Hybrid",
      ["Immutable contract logic guarding high-value funds", "Approval-threshold invariants need formal verification", "Off-chain monitors need continuous security scanning", "Mandatory audit trail for sensitive actions"],
      "Contract logic that gates large withdrawals cannot be patched cheaply once deployed, so the on-chain part needs V-Model verification with traceable test levels and formal invariants. The off-chain signer dashboards and anomaly monitors change more often and benefit from DevSecOps pipelines with static analysis and fuzzing. Pure Agile gives too little assurance for irreversible fund movements."),
    S(DEFI, "Oracle price feed validation and automated liquidation",
      [("Functional", "The protocol shall recompute every borrower's collateralization ratio on each block using the approved oracle price feed."),
       ("Security", "The liquidation engine shall compare the primary oracle price with a 30-minute time-weighted average and pause liquidations when the deviation exceeds 2.5%."),
       ("Reliability", "If the oracle price is older than 10 minutes, the protocol shall block new borrows and liquidations until a fresh price is received."),
       ("Functional", "When a position falls below the liquidation threshold, liquidators shall be allowed to repay up to 50% of the debt in exchange for discounted collateral."),
       ("Auditability", "Changes to collateral factors and liquidation thresholds shall be logged on-chain with the proposer identity and timestamp.")],
      "Liquidations are driven by oracle prices, so delayed or manipulated prices can wrongly liquidate solvent users or leave the pool with bad debt. Cross-checking against a time-weighted average, handling stale feeds and logging parameter changes bound that risk.",
      "V-Model",
      ["Correctness-critical liquidation arithmetic", "Well-understood, stable liquidation rules", "Need for traceable test levels and evidence", "High cost of failure: bad debt and user losses"],
      "The liquidation rules are well defined and rarely change, while a mistake causes direct financial loss. The V-Model pairs each requirement with a test level - invariant specification, unit and property tests, integration tests against oracle-failure scenarios - and yields the traceable evidence auditors expect. Agile iteration adds little for stable rules, and Spiral's risk loops are unnecessary for a known mechanism."),
    S(DEFI, "Block-by-block interest rate recalculation with user slippage protection",
      [("Functional", "The pool contract shall recalculate borrow and supply rates on every block from the current utilization ratio."),
       ("Usability", "The interface shall display the projected effective rate before the user confirms a transaction."),
       ("Security", "Borrow transactions shall accept a maximum-rate parameter and revert if the on-chain rate exceeds it at execution."),
       ("Auditability", "Changes to rate-curve parameters shall record the proposer identity and timestamp on-chain.")],
      "Rates float with utilization and can move within the same block, so users are surprised unless they see the rate first and can cap it. Bounded execution and parameter logging keep behaviour predictable and auditable.",
      "Agile-V-Model Hybrid",
      ["Rate-curve tuning expected to evolve with market data", "Core rate math needs rigorous verification", "Interface benefits from short feedback loops", "Parameter changes need audit evidence"],
      "The user-facing rate display and curve parameters will be tuned frequently, which suits short Agile iterations, but the on-chain rate and slippage logic is financial code that needs V-Model style verification before each release. A hybrid lets the team iterate on the interface and parameters while holding contract changes to formal test gates."),
    S(DEFI, "Position and health-factor dashboard for lenders and borrowers",
      [("Functional", "The dashboard shall show each user's supplied and borrowed balances, current health factor and accrued interest, refreshed at least every 30 seconds."),
       ("Usability", "The dashboard shall show a colour-coded liquidation warning when the health factor falls below 1.5."),
       ("Performance", "The dashboard shall load the position summary within 2 seconds for 95% of requests."),
       ("Privacy", "The dashboard shall display only the connected wallet's positions and shall not store wallet-to-identity mappings in analytics logs.")],
      "Users need timely visibility of their positions to avoid liquidation. This is a read-only off-chain view, so the main concerns are freshness, clear risk signals and not leaking wallet-linked personal data.",
      "Agile",
      ["Read-only off-chain application that moves no funds", "Requirements expected to change with user feedback", "Low consequence of failure compared with on-chain logic", "Stakeholders available for frequent feedback"],
      "This component reads data and never moves funds, so a defect is recoverable with a quick release. Requirements will keep changing as users react to layout and metrics, which Agile's short sprints and continuous feedback handle well; heavyweight verification would add cost without reducing material risk."),
    # ------------------------------------------------------------------ Cross-Border Payments
    S(XB, "Real-time FX quoting and stablecoin settlement for remittances",
      [("Functional", "The system shall present a firm FX quote, the fee and the final received amount on one screen and keep the quote valid for 30 seconds."),
       ("Functional", "The system shall request a new quote when the customer confirms after the 30-second validity has expired."),
       ("Performance", "Settlement from sender confirmation to receiver payout shall complete within 5 minutes for 95% of transfers."),
       ("Integration", "The system shall convert funds to the USD stablecoin, transfer them on-chain and trigger payout through the licensed receiving partner, recording each leg under one shared transfer ID.")],
      "Customers must not be surprised by rate or fees, so the quote is firm and time-boxed. Settlement speed is the product's selling point, and a shared transfer ID across conversion, on-chain and payout legs makes every payment traceable end to end.",
      "Agile-V-Model Hybrid",
      ["Corridors, partners and pricing rules change frequently", "Settlement logic moves customer funds", "Several external partners need integration testing", "Quote and fee screens improve with feedback"],
      "New corridors and partners arrive continuously, so delivery should be incremental, but the quote, conversion and payout logic handles customer money and needs traceable V-Model style verification including partner integration tests. Agile sprints with mandatory verification gates for the settlement core balance speed and assurance."),
    S(XB, "Travel Rule data exchange between providers before settlement",
      [("Regulatory", "For transfers above the corridor threshold, the system shall collect originator and beneficiary information and send it to the receiving provider before settlement."),
       ("Security", "The system shall exchange Travel Rule data only with providers on the approved counterparty list, over mutually authenticated encrypted channels."),
       ("Regulatory", "The system shall hold a transfer in a pending state until the counterparty acknowledges the data, releasing it only after acknowledgement or a compliance override."),
       ("Auditability", "The system shall retain each exchanged payload and acknowledgement for the legally required period and make them retrievable by auditors.")],
      "Regulators require identifying data to travel with larger transfers, but that data is sensitive. Restricting counterparties, authenticating channels, blocking settlement until acknowledgement and retaining evidence satisfy the rule while protecting personal data.",
      "DevSecOps-V-Model Hybrid",
      ["High regulatory criticality with per-corridor thresholds", "Sensitive personal data in transit", "Counterparty protocols and rules keep evolving", "Compliance sign-off needed before release"],
      "Thresholds, counterparties and message formats change as regulators and partners update rules, which calls for continuous delivery with automated security checks. Because errors create compliance breaches and expose personal data, each release also needs V-Model validation gates and compliance sign-off, so a DevSecOps pipeline with formal approval gates fits best."),
    S(XB, "Automatic refunds and daily reconciliation of failed transfers",
      [("Functional", "The system shall refund the sender automatically within 24 hours when a transfer cannot be completed and shall notify the sender of the outcome."),
       ("Functional", "The system shall reconcile on-chain transfers with partner bank ledgers every day and list unmatched items with their amounts and transfer IDs."),
       ("Operational", "Unmatched items shall be escalated to operations the same day, with an owner assigned and status tracked until resolved."),
       ("Security", "Refund and reconciliation jobs shall run under least-privilege service accounts, and every automated action shall be logged immutably.")],
      "Automatic refunds and daily reconciliation protect customers and catch discrepancies early. Because these jobs move and compare money without human review, least-privilege access and tamper-evident logs limit the impact of mistakes or misuse.",
      "DevSecOps",
      ["Automated back-office jobs that run continuously", "Security risk on automated money movement", "Frequent small changes to partner file formats", "Strong monitoring and rollback needs"],
      "These are automated, always-running jobs where small, frequent changes such as new partner file formats and rule tweaks are expected. DevSecOps provides CI/CD with automated tests, secret scanning, monitoring and fast rollback. The logic is well understood, so heavy phase-gated verification is not needed."),
    S(XB, "Remittance mobile app with multilingual status tracking",
      [("Usability", "The app shall be available in the local languages of each supported corridor and let users switch language at any time."),
       ("Functional", "The app shall show a clear status (quoted, sent, in transit, paid out, refunded) for every transfer."),
       ("Functional", "The app shall send SMS status updates when the user's data connection is unavailable."),
       ("Performance", "The transfer screen shall load within 3 seconds on a 3G connection on a low-end device.")],
      "Many users have limited digital literacy and weak connectivity, so language support, plain status messages and SMS fallbacks drive adoption. These are mostly interface requirements that need testing with real users.",
      "Agile",
      ["Interface-centred work with unclear user preferences", "Requirements discovered through user testing", "Money movement sits in backend services, not the app", "Stakeholders available for frequent feedback"],
      "Usability needs for new corridors only emerge through contact with real users, so short iterations with frequent releases and feedback are the most effective way to converge. The app does not itself settle funds, so lightweight verification is acceptable and Agile delivers value fastest."),
    # ------------------------------------------------------------------ KYC / AML
    S(KYC, "Customer onboarding identity tiers and transfer limits",
      [("Compliance", "The platform shall block off-platform transfers from any linked wallet until the customer has completed identity verification."),
       ("Functional", "The platform shall apply daily and monthly outbound transfer limits according to the customer's verification tier."),
       ("Security", "Identity documents and verification results shall be encrypted at rest and accessible only to roles with a documented compliance purpose."),
       ("Auditability", "Every tier change and limit override shall be recorded with the actor, reason and timestamp.")],
      "Policy ties the ability to move funds out to verified identity and tiers. Enforcing this in the transfer path, protecting identity data and logging changes satisfy compliance while limiting exposure of personal data.",
      "DevSecOps-V-Model Hybrid",
      ["High regulatory criticality for AML controls", "Sensitive personal data", "Tier thresholds change with regulation", "Auditable approvals required"],
      "AML controls are legally mandatory and process sensitive data, so releases need documented validation and sign-off, which V-Model gates provide. Limits and tiers change as rules evolve, which a DevSecOps pipeline with automated security scanning supports without long release cycles."),
    S(KYC, "Sanctions screening of wallet addresses",
      [("Compliance", "The system shall screen wallet addresses against current sanctions lists at onboarding and at least daily thereafter."),
       ("Compliance", "The system shall freeze outbound transfers for an address with a positive match and open a case for manual compliance review."),
       ("Operational", "The system shall load updated sanctions lists within one hour of publication and record the list version used for each screening."),
       ("Performance", "Screening of an onboarding address shall complete within 2 seconds for 99% of checks.")],
      "Sanctions lists change often and a missed match is a serious breach. Frequent automated updates, version recording and fast screening ensure decisions use current data and can be evidenced later.",
      "DevSecOps",
      ["Frequently changing external data", "Continuous deployment of rule and data updates", "Automated regression testing of match accuracy", "Well-understood logic with moderate scope"],
      "The screening logic is well understood, but the data and matching rules change constantly. DevSecOps' continuous integration, automated regression tests for match accuracy and monitoring let the team ship updates quickly and safely, whereas a phase-heavy model would lag behind list changes."),
    S(KYC, "Suspicious activity reporting to the regulator",
      [("Regulatory", "The system shall let compliance officers compile and file suspicious activity reports in the regulator-prescribed format before the legal deadline."),
       ("Functional", "Each report shall require review and sign-off by the compliance officer before submission."),
       ("Auditability", "The system shall keep each report, its supporting evidence and the sign-off record for the legally required retention period."),
       ("Operational", "The system shall alert the compliance officer when a case is 48 hours from its filing deadline.")],
      "Reporting format and workflow are fixed by the regulator, and every filing needs accountable sign-off. Deadline alerts and retained evidence reduce the risk of late or unsupported reports.",
      "Waterfall",
      ["Requirements fixed by the regulator and rarely changing", "Formal sign-offs and documentation required", "Small, well-defined scope", "Low technical uncertainty"],
      "The report format and workflow are prescribed and change rarely, so requirements can be fully specified up front. Waterfall's documented phases and formal approvals match the compliance sign-off culture, and low uncertainty means little benefit from iterative discovery."),
    S(KYC, "Reusable verifiable identity credentials for customer verification",
      [("Functional", "The platform shall accept a customer's verifiable identity credential from a trusted issuer and use it to satisfy the identity-verification step."),
       ("Security", "The platform shall verify the credential's issuer signature and revocation status before accepting it."),
       ("Privacy", "The platform shall request only the credential attributes required for the customer's verification tier."),
       ("Compliance", "Where the regulator does not accept credential-based verification, the platform shall fall back to standard document verification.")],
      "Reusable credentials could remove repeated checks, but standards and regulatory acceptance are unsettled. The requirements therefore stress signature and revocation checks, minimal disclosure and a safe fallback.",
      "Spiral",
      ["High technical and regulatory uncertainty", "Evolving identity standards", "Need to prototype with issuers and regulators", "Risk reduction before large investment"],
      "Standards, issuers and regulatory acceptance are all uncertain, so the main risk is building the wrong thing. Spiral's repeated cycles of risk analysis, prototyping and stakeholder evaluation let the team test assumptions with issuers and regulators before committing to a full build."),
    # ------------------------------------------------------------------ Stablecoin
    S(STB, "Mint and burn authorization tied to reserve confirmation",
      [("Security", "The mint function shall execute only after the matching reserve deposit has been confirmed by two independent authorized officers."),
       ("Functional", "The system shall burn redeemed tokens on-chain before fiat release is initiated."),
       ("Functional", "Total token supply shall never exceed confirmed reserves, and the contract shall reject any mint that would break this invariant."),
       ("Auditability", "Every mint and burn shall emit an event referencing the reserve confirmation ID and the approvers.")],
      "The 1:1 backing promise is the core of the product, so the contract must enforce it rather than rely on process alone. Dual confirmation, burn-before-release and an explicit supply invariant make breaches detectable and preventable.",
      "V-Model",
      ["Core invariant (supply <= reserves) needs proof-level assurance", "Stable, well-defined issuance rules", "Irreversible on-chain consequences", "Regulator expects traceable verification evidence"],
      "Mint and burn rules are stable and defined by policy, and a flaw would break the token's backing irreversibly. The V-Model pairs each requirement with a test or formal check, producing the traceable verification evidence regulators and auditors expect, which outweighs the flexibility of iterative models here."),
    S(STB, "Monthly reserve attestation publication",
      [("Regulatory", "The system shall produce the monthly reserve attestation summary in the regulator-prescribed format."),
       ("Functional", "The summary shall be published to token holders and delivered to the regulator within five business days of month end."),
       ("Auditability", "The system shall retain the attestation, source balances and publication logs for the required retention period."),
       ("Security", "Only authorized finance officers shall be able to approve a report for publication, and a published report shall be immutable.")],
      "The report content, format and deadline are set by the regulator. Controlled approval, immutability and retention make the published figures trustworthy and defensible.",
      "Waterfall",
      ["Format and deadline fixed by regulation", "Stable requirements with formal approvals", "Low technical risk", "Documentation-heavy deliverable"],
      "Requirements are fixed by the regulator and unlikely to change, and approvals and documentation are central. A sequential Waterfall plan with formal reviews at each phase fits and keeps overhead low for a well-understood reporting feature."),
    S(STB, "Depeg circuit breaker and redemption continuity",
      [("Functional", "The system shall pause minting when the market price stays more than 1% from the peg for 15 minutes and alert the risk committee."),
       ("Functional", "The system shall continue to honour redemptions at par within one business day while minting is paused."),
       ("Security", "Only the risk committee, through multi-party approval, shall be able to lift a pause."),
       ("Reliability", "The price-monitoring service shall be available 99.9% of the time and shall alert operators if a price source fails.")],
      "The coin's credibility depends on reacting quickly to a depeg while still letting holders exit at par. Multi-party control of the pause and monitoring that alerts on failure keep the safeguard both effective and hard to abuse.",
      "DevSecOps-V-Model Hybrid",
      ["Safety control must work under market stress", "Monitoring and thresholds are tuned over time", "On-chain pause logic is hard to patch", "Strict approval and audit requirements"],
      "The pause logic is high-consequence on-chain code needing rigorous V-Model verification including stress scenarios, while the monitoring services and thresholds are tuned continuously and benefit from DevSecOps automation, observability and rapid secure releases. The hybrid covers both layers."),
    S(STB, "Algorithmic supply-adjustment research module",
      [("Functional", "The prototype shall adjust a supplementary supply parameter in response to price deviation according to a configurable control rule."),
       ("Reliability", "The prototype shall be simulated against historical and synthetic stress scenarios, reporting peg deviation and reserve drawdown for each run."),
       ("Security", "The mechanism shall have bounded adjustment limits so that one update cannot change supply by more than a configured percentage."),
       ("Governance", "Production use shall require explicit risk-committee approval based on simulation evidence.")],
      "The behaviour of an algorithmic mechanism under stress is not well understood, so the requirements prioritise simulation, bounded changes and a human approval gate before any production use.",
      "Spiral",
      ["Poorly understood behaviour under stress", "Research-driven, exploratory scope", "Need for repeated prototyping and risk evaluation", "Go/no-go decision depends on evidence"],
      "The central risk is technical: nobody yet knows whether the mechanism is stable. Spiral lets the team build a prototype, simulate, assess risk and decide whether to proceed in each loop, avoiding a large commitment before the risks are understood."),
    # ------------------------------------------------------------------ Crypto Custody
    S(CUS, "Hot and cold wallet key management",
      [("Security", "Private keys shall be generated and stored only inside hardware security modules and shall never appear in plaintext outside them."),
       ("Security", "Signing shall use multi-party computation so that no single person or server holds a complete key."),
       ("Functional", "The system shall keep at least 95% of client assets in cold storage and alert operations when the hot wallet balance exceeds its daily limit."),
       ("Operational", "The system shall support key rotation and recovery procedures that are rehearsed at least twice a year.")],
      "Loss or theft of keys means irreversible loss of client assets, so key handling must avoid any single point of compromise. Cold-storage ratios, hot-wallet limits and rehearsed recovery keep exposure small and recoverable.",
      "V-Model",
      ["Key loss means irreversible loss of client assets", "Stable, well-understood security architecture", "Need for formal test and audit evidence", "Low need for rapid feature change"],
      "The key-management architecture is stable and its failure modes are catastrophic, so it should be specified, verified and validated level by level, with documented evidence for auditors. The V-Model's paired test levels, including recovery drills, fit better than iterative models that optimise for change."),
    S(CUS, "Withdrawal address whitelisting and cooling period",
      [("Security", "The system shall allow withdrawals only to whitelisted addresses."),
       ("Security", "A newly added address shall not receive funds until a 24-hour cooling period has passed."),
       ("Functional", "The system shall notify the client by email and in the app when an address is added or removed."),
       ("Security", "Withdrawals above the approved value shall require approval from two authorized operators.")],
      "Attackers who take over an account usually try to add their own address and withdraw at once. A cooling period, client notification and dual approval for large amounts give the client time to react.",
      "Agile-V-Model Hybrid",
      ["Policy limits and client flows expected to evolve", "Controls protect client funds and need verification", "Customer-facing flows benefit from iteration", "Approval workflows need audit evidence"],
      "Client-facing flows and limits will be adjusted as fraud patterns and customer feedback emerge, which suits short iterations. The controls themselves protect client funds, so each increment must pass defined verification gates and keep audit evidence. An Agile cadence with V-Model style gates on the security controls meets both needs."),
    S(CUS, "Internal signing service API for the trading desk",
      [("Availability", "The signing API shall be available 24 hours a day with at least 99.95% uptime."),
       ("Security", "The API shall reject any request that violates policy limits, destination rules or caller permissions."),
       ("Auditability", "The API shall log every request, decision and caller identity to tamper-evident storage."),
       ("Performance", "The API shall return a signing decision within 500 milliseconds for 95% of requests.")],
      "The trading desk depends on this service all day, and it is the gate through which funds move. It must therefore be always on, strictly policy-driven and fully logged.",
      "DevSecOps",
      ["Always-on internal service", "Frequent policy rule updates", "Security scanning and monitoring are central", "Clear, well-understood API contract"],
      "The interface is well understood but policy rules and limits change often, and the service must stay available and secure. DevSecOps supplies automated testing, dependency and secret scanning, monitored deployments and rapid rollback, which suit a continuously operated internal service."),
    S(CUS, "Retail mobile wallet app",
      [("Functional", "The app shall let clients view balances, receive funds and request withdrawals."),
       ("Security", "The app shall require biometric or PIN authentication and lock after 5 minutes of inactivity."),
       ("Usability", "A client shall be able to complete a withdrawal request in no more than 4 steps."),
       ("Performance", "Balance screens shall refresh within 3 seconds of opening the app.")],
      "Retail clients judge the service by how simple and safe the app feels. Strong local authentication and a short withdrawal flow serve both goals, and the business expects to keep changing screens as it learns.",
      "Agile",
      ["Frequent screen and feature changes expected", "User feedback drives priorities", "Custody security sits in backend services", "Short feedback loops possible"],
      "Features and screens are expected to change often in response to feedback, and the app sits in front of backend controls that carry the main security risk. Agile's short iterations and continuous user feedback are the most effective way to refine the experience."),
    # ------------------------------------------------------------------ Tokenized Trade Finance
    S(TF, "Letter of credit management across consortium banks",
      [("Functional", "The ledger shall record each letter of credit with issuing bank, beneficiary, amount, terms and expiry date."),
       ("Compliance", "Document checks shall follow the consortium's agreed documentary-credit rules, and any change to those rules shall require approval from every member bank."),
       ("Auditability", "Each acceptance, rejection or amendment shall be recorded with the acting bank and timestamp."),
       ("Security", "Only authenticated staff of the parties to a letter of credit shall be able to view or act on it.")],
      "The process is long established and shared by several banks, so the system must reproduce the agreed rules faithfully, record every action and restrict visibility to the parties involved.",
      "Waterfall",
      ["Stable, long-established business rules", "Approval from every consortium bank for changes", "Heavy documentation and legal review", "Low uncertainty"],
      "The rules are stable and any change needs unanimous approval and legal review, so requirements can be fixed early and delivered through formally approved phases. Waterfall's documentation and sign-off structure matches this governance, and little is gained from iterative discovery."),
    S(TF, "Invoice tokenization and fractional investor financing",
      [("Functional", "The platform shall issue a token for an approved invoice that can be funded in portions by multiple investors."),
       ("Functional", "The platform shall distribute repayment to investors in proportion to their funded share when the importer pays."),
       ("Security", "A token shall not be transferable to an investor who has not passed eligibility checks."),
       ("Auditability", "The platform shall keep a complete ledger of funding, transfers and repayments for each invoice token.")],
      "Splitting an invoice among investors creates obligations about who funded what and who is repaid. Proportional repayment, eligibility checks and a full ledger keep the product fair and auditable.",
      "Agile-V-Model Hybrid",
      ["New product expected to change after pilot feedback", "Token and repayment logic handles investor money", "Eligibility rules need verified enforcement", "Complete audit ledger needed"],
      "Because the product is new and will be adjusted after pilots, short iterations help the team learn quickly. The token, payment and eligibility logic handles investor funds, so those components need V-Model style verification before each release. An Agile cadence with verification gates on the financial core fits."),
    S(TF, "Bill of lading anchoring and shipment events from logistics feeds",
      [("Functional", "The system shall hash each bill of lading and anchor the hash on the ledger when it is received."),
       ("Integration", "The system shall ingest shipment events from logistics partner feeds and map them to a common event format."),
       ("Reliability", "The system shall flag events that conflict with earlier events or come from an unverified source and hold related payments for review."),
       ("Security", "Only authorized parties shall be able to submit shipment events, and the source of every event shall be recorded.")],
      "Payment decisions may depend on shipment data from feeds whose formats and reliability vary. Anchoring documents, normalising events and holding payments when data conflicts reduce the risk of acting on bad information.",
      "Spiral",
      ["Unreliable, inconsistent external data feeds", "Unproven integrations with several partners", "Need to prototype and evaluate feed quality", "Risk of payments triggered by bad data"],
      "Feed quality and partner behaviour are unknown until tried, so the main risk is technical and integration-related. Spiral's loops of prototyping against real feeds, risk assessment and re-planning let the team discover and contain data-quality problems before committing to a full design."),
    S(TF, "Consortium settlement and dispute handling",
      [("Functional", "The system shall settle payment between banks once the required documents are accepted."),
       ("Functional", "The system shall open a dispute case when a discrepancy is raised and block settlement until the case is resolved."),
       ("Auditability", "Each dispute decision shall be recorded with the reviewer, reason and timestamp."),
       ("Reliability", "Settlement shall be atomic: either all legs of a payment complete or none do.")],
      "Money moves between banks only after documents are accepted, and discrepancies must stop settlement. Atomic settlement and recorded decisions prevent half-completed payments and unexplained outcomes.",
      "V-Model",
      ["Settlement correctness is critical", "Well-defined, stable settlement and dispute rules", "Need for thorough integration and acceptance testing", "Records needed for audit"],
      "Settlement and dispute rules are well defined and errors move money incorrectly between banks. The V-Model's matching test levels - unit, integration across banks, and acceptance against the agreed process - give systematic verification and traceability for this high-consequence logic."),
    # ------------------------------------------------------------------ CBDC
    S(CBDC, "Offline retail payments for low-connectivity regions",
      [("Functional", "The wallet shall allow payments of limited value between two devices without a network connection."),
       ("Security", "Offline balances shall be held in secure hardware or equivalent protection and bounded by a per-device offline limit."),
       ("Security", "The system shall detect double-spending when devices reconnect and apply defined recovery rules."),
       ("Usability", "The wallet shall show offline balance, pending offline transactions and synchronization status.")],
      "Offline value transfer cannot be checked against a live ledger, so the risk of double spending must be bounded by limits, secure storage and reconciliation rules. Clear status display helps users trust a mechanism they cannot see being confirmed.",
      "Spiral",
      ["Technology and risk model still being explored", "Pilot-driven learning expected", "Security assumptions need prototyping and testing", "Go/no-go depends on pilot evidence"],
      "The policy notes say the technology and risk model are still under exploration through pilots. Spiral's cycles of prototyping, risk analysis and evaluation suit that uncertainty, letting each pilot refine the security assumptions before the design is committed."),
    S(CBDC, "Interbank settlement through the central bank RTGS system",
      [("Integration", "The system shall settle intermediary obligations through the central bank's RTGS interface according to the published specification."),
       ("Functional", "The system shall submit settlement instructions in the central bank's prescribed message format and reconcile the confirmations."),
       ("Availability", "The settlement interface shall meet 99.99% availability."),
       ("Auditability", "All settlement instructions and confirmations shall be retained and reportable to the central bank.")],
      "The interface and its specification are fixed by the central bank, so the work is to implement it faithfully, keep it available and retain complete records for reporting.",
      "Waterfall",
      ["Interface specification fixed by the central bank", "Formal approval for any change", "Stable requirements", "Low design uncertainty"],
      "The RTGS interface is specified externally and changes only through formal approval, so requirements are stable and knowable up front. A sequential Waterfall plan with documented reviews and acceptance fits, and iterative discovery would add little."),
    S(CBDC, "Programmable conditional payments such as time-limited subsidies",
      [("Functional", "The platform shall let an authorized issuer attach conditions, such as an expiry date or merchant category, to a payment."),
       ("Security", "Conditions shall be enforced at settlement and shall not be alterable by the recipient."),
       ("Functional", "Unspent conditional funds shall return to the issuer after expiry."),
       ("Auditability", "The platform shall record the conditions, the issuer and the outcome of each conditional payment.")],
      "Conditional payments let policymakers target spending, but the exact conditions they need are still being studied. Enforcing conditions at settlement and recording outcomes keeps the feature trustworthy while allowing rule types to grow.",
      "Agile-V-Model Hybrid",
      ["Policy needs still being studied", "Condition types expected to grow", "Enforcement logic touches public money", "Auditable outcomes required"],
      "New condition types will keep emerging as policymakers experiment, which suits incremental delivery, while the enforcement and return-of-funds logic handles public money and needs V-Model style verification at each increment. A hybrid delivers new conditions quickly without weakening assurance on the core."),
    S(CBDC, "Wallet tiers, transaction limits and privacy thresholds",
      [("Functional", "Each wallet shall have a maximum balance and a daily transaction limit according to its identity tier."),
       ("Privacy", "Payments below the privacy threshold shall not expose the payer's identity to merchants or intermediaries."),
       ("Compliance", "Payments above the threshold shall be traceable to authorized authorities under lawful process."),
       ("Performance", "The platform shall process at least 2,000 transactions per second at peak with 99.99% availability.")],
      "Policy balances privacy for small payments against traceability for large ones, with limits tied to identity tier. The requirements encode those rules precisely and add the stated performance and availability targets.",
      "V-Model",
      ["Privacy and traceability rules must be exactly correct", "Stable, policy-defined limits", "Strict performance and availability targets to validate", "Evidence needed for the central bank"],
      "The limit and privacy rules are policy-defined and must be implemented exactly, and the throughput and availability targets must be demonstrated. The V-Model's requirement-to-test traceability, including performance and compliance validation levels, provides the evidence a central bank will expect."),
]
__END_OF_FILE__
cat > generate_dataset.py << '__END_OF_FILE__'
"""Build the fine-tuning datasets from seed_data.py.

    python generate_dataset.py            -> train_stage1.jsonl  (LLM 1: requirements)
                                             train_stage2.jsonl  (LLM 2: SDLC only)

The prompts come from pipeline.py and the evidence comes from running the real retriever over
sample_docs/, so the training prompts are exactly what the models will see at run time.
If you change sample_docs/ or the prompts, re-run this script and re-train.
"""
from __future__ import annotations

import argparse
import json
import os
from collections import Counter

from pipeline import SDLC_LABELS, build_index, load_documents, retrieve_evidence, stage1_messages, stage2_messages
from seed_data import SEEDS


def _record(system: str, user: str, assistant: dict) -> str:
    return json.dumps({"messages": [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
        {"role": "assistant", "content": json.dumps(assistant, ensure_ascii=False)},
    ]}, ensure_ascii=False)


def build_examples(docs_dir: str, top_k: int = 4):
    retriever = build_index(load_documents(docs_dir))
    stage1, stage2, problems = [], [], []
    for n, seed in enumerate(SEEDS, 1):
        tag = f"seed {n} ({seed['domain']}: {seed['functionality']})"
        if seed["sdlc"] not in SDLC_LABELS:
            problems.append(f"{tag}: unknown SDLC label {seed['sdlc']!r}")
        evidence = retrieve_evidence(retriever, seed["domain"], seed["functionality"], top_k)
        if not evidence:
            problems.append(f"{tag}: retrieval found no evidence")
        reqs = [{"id": f"REQ-{i:03d}", "statement": st, "category": cat}
                for i, (cat, st) in enumerate(seed["requirements"], 1)]
        for r in reqs:
            if " shall " not in f" {r['statement']} ":
                problems.append(f"{tag}: {r['id']} does not use 'shall'")

        sys1, usr1 = stage1_messages(seed["domain"], seed["functionality"], evidence)
        stage1.append(_record(sys1, usr1, {"requirements": reqs, "reasoning": seed["reasoning"]}))

        sys2, usr2 = stage2_messages(seed["domain"], reqs, evidence)
        stage2.append(_record(sys2, usr2, {"recommended_sdlc": seed["sdlc"], "key_factors": seed["factors"],
                                           "justification": seed["justification"]}))
    return stage1, stage2, problems


def main() -> None:
    here = os.path.dirname(os.path.abspath(__file__))
    p = argparse.ArgumentParser(description="Generate fine-tuning datasets")
    p.add_argument("--docs", default=os.path.join(here, "sample_docs"))
    p.add_argument("--out-dir", default=here)
    args = p.parse_args()

    stage1, stage2, problems = build_examples(args.docs)
    if problems:
        raise SystemExit("Dataset problems:\n  " + "\n  ".join(problems))
    for name, rows in (("train_stage1.jsonl", stage1), ("train_stage2.jsonl", stage2)):
        with open(os.path.join(args.out_dir, name), "w", encoding="utf-8") as f:
            f.write("\n".join(rows) + "\n")
        print(f"[ok] wrote {len(rows)} examples to {name}")
    print("SDLC label distribution:", dict(Counter(s["sdlc"] for s in SEEDS)))


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > finetune_llm.py << '__END_OF_FILE__'
"""Fine-tune ONE of the two models and export it for Ollama.   (needs an NVIDIA GPU - e.g. free Colab T4)

    python generate_dataset.py                 # once: builds train_stage1.jsonl / train_stage2.jsonl
    python finetune_llm.py --stage 1           # LLM 1: Llama-3.2-3B -> requirements   => exports/blockchain-req-llm/
    python finetune_llm.py --stage 2           # LLM 2: Qwen2.5-3B   -> SDLC only      => exports/blockchain-sdlc-llm/

Each export folder holds `model.Q4_K_M.gguf` + a `Modelfile`. Copy the `exports/` folder to the machine that
runs Ollama and run `python install_models.py` there (see README).

Design note: the text format used for training (FORMATS below) and the TEMPLATE written into the Modelfile are
defined side by side, so the model is trained on exactly the prompt format Ollama will send it.
"""
from __future__ import annotations

import argparse
import glob
import json
import os
import shutil

MAX_SEQ_LENGTH = 3072

STAGES = {
    1: {"base": "unsloth/Llama-3.2-3B-Instruct", "family": "llama", "data": "train_stage1.jsonl", "name": "blockchain-req-llm"},
    2: {"base": "unsloth/Qwen2.5-3B-Instruct", "family": "qwen", "data": "train_stage2.jsonl", "name": "blockchain-sdlc-llm"},
}

LLAMA_TEMPLATE = """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""

QWEN_TEMPLATE = """{{ if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ .Response }}<|im_end|>"""

FORMATS = {
    "llama": {
        "block": lambda role, content: f"<|start_header_id|>{role}<|end_header_id|>\n\n{content}<|eot_id|>",
        "generation_prefix": "<|start_header_id|>assistant<|end_header_id|>\n\n",
        "instruction_part": "<|start_header_id|>user<|end_header_id|>\n\n",
        "response_part": "<|start_header_id|>assistant<|end_header_id|>\n\n",
        "stops": ["<|start_header_id|>", "<|end_header_id|>", "<|eot_id|>"],
        "template": LLAMA_TEMPLATE,
    },
    "qwen": {
        "block": lambda role, content: f"<|im_start|>{role}\n{content}<|im_end|>\n",
        "generation_prefix": "<|im_start|>assistant\n",
        "instruction_part": "<|im_start|>user\n",
        "response_part": "<|im_start|>assistant\n",
        "stops": ["<|im_start|>", "<|im_end|>"],
        "template": QWEN_TEMPLATE,
    },
}


# ----------------------------------------------------------------------------- pure helpers (unit-tested, no GPU needed)
def render_example(messages: list, family: str) -> str:
    """Full training text for one {system, user, assistant} example (no BOS token: the tokenizer adds it)."""
    block = FORMATS[family]["block"]
    return "".join(block(m["role"], m["content"]) for m in messages)


def render_prompt(system: str, user: str, family: str) -> str:
    fmt = FORMATS[family]
    return fmt["block"]("system", system) + fmt["block"]("user", user) + fmt["generation_prefix"]


def modelfile_text(gguf_name: str, family: str, num_ctx: int = 4096) -> str:
    fmt = FORMATS[family]
    lines = [f"FROM ./{gguf_name}", f'TEMPLATE """{fmt["template"]}"""']
    lines += [f'PARAMETER stop "{s}"' for s in fmt["stops"]]
    lines += ["PARAMETER temperature 0.2", f"PARAMETER num_ctx {num_ctx}"]
    return "\n".join(lines) + "\n"


def find_gguf(work_dir: str) -> str:
    """Locate the quantised GGUF that Unsloth wrote (its folder/file naming differs between versions)."""
    candidates = []
    for d in (work_dir, work_dir + "_gguf"):
        candidates += glob.glob(os.path.join(d, "**", "*.gguf"), recursive=True)
    candidates = [c for c in candidates if not any(t in os.path.basename(c).lower() for t in ("f16", "bf16", "f32"))]
    if not candidates:
        raise FileNotFoundError(f"No quantised .gguf file found under {work_dir}[_gguf]. Look inside those folders manually.")
    q4 = [c for c in candidates if "q4_k_m" in os.path.basename(c).lower()]
    return (q4 or candidates)[0]


# ----------------------------------------------------------------------------- training
def make_trainer(SFTConfig, SFTTrainer, model, tokenizer, dataset, cfg: dict):
    """Build an SFTTrainer across TRL versions (argument names moved between releases)."""
    attempts = [
        ({"dataset_text_field": "text", "max_seq_length": MAX_SEQ_LENGTH}, {"processing_class": tokenizer}),
        ({"dataset_text_field": "text", "max_length": MAX_SEQ_LENGTH}, {"processing_class": tokenizer}),
        ({}, {"tokenizer": tokenizer, "dataset_text_field": "text", "max_seq_length": MAX_SEQ_LENGTH}),
    ]
    last = None
    for cfg_extra, trainer_kw in attempts:
        try:
            return SFTTrainer(model=model, train_dataset=dataset, args=SFTConfig(**cfg, **cfg_extra), **trainer_kw)
        except TypeError as e:
            last = e
    raise RuntimeError(f"Could not build SFTTrainer with the installed TRL version: {last}")


def main() -> None:
    p = argparse.ArgumentParser(description="Fine-tune one stage and export a GGUF + Modelfile for Ollama")
    p.add_argument("--stage", type=int, choices=[1, 2], required=True)
    p.add_argument("--epochs", type=float, default=3, help="2-3 is plenty for a small dataset; more will over-fit")
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--base-model", default=None, help="override the Hugging Face base model")
    p.add_argument("--data", default=None, help="override the training JSONL")
    p.add_argument("--name", default=None, help="override the Ollama model name")
    p.add_argument("--out", default="exports")
    args = p.parse_args()

    cfg = STAGES[args.stage]
    family, base = cfg["family"], args.base_model or cfg["base"]
    data_path, name = args.data or cfg["data"], args.name or cfg["name"]
    fmt = FORMATS[family]
    if not os.path.exists(data_path):
        raise SystemExit(f"{data_path} not found - run `python generate_dataset.py` first.")

    # Heavy imports are here (not at the top) so `--help` and the unit tests work without a GPU.
    from unsloth import FastLanguageModel   # must be imported before trl / transformers
    import torch
    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer

    if not torch.cuda.is_available():
        raise SystemExit("No GPU found. Use a GPU runtime (Colab: Runtime > Change runtime type > T4 GPU).")

    print(f"[1/5] Loading base model {base} in 4-bit ...")
    model, tokenizer = FastLanguageModel.from_pretrained(model_name=base, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=42,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth")

    print(f"[2/5] Preparing {data_path} ...")
    dataset = load_dataset("json", data_files=data_path, split="train")
    dataset = dataset.map(lambda ex: {"text": render_example(ex["messages"], family)}, remove_columns=dataset.column_names)
    longest = max(len(tokenizer(t)["input_ids"]) for t in dataset["text"])
    print(f"      {len(dataset)} examples, longest = {longest} tokens (limit {MAX_SEQ_LENGTH})")
    if longest > MAX_SEQ_LENGTH:
        print("      WARNING: some examples are longer than the limit and will be truncated - lower top-k or raise MAX_SEQ_LENGTH.")

    print("[3/5] Training (QLoRA) ...")
    use_bf16 = torch.cuda.is_bf16_supported()
    trainer = make_trainer(SFTConfig, SFTTrainer, model, tokenizer, dataset, dict(
        per_device_train_batch_size=1, gradient_accumulation_steps=4, warmup_steps=3,
        num_train_epochs=args.epochs, learning_rate=args.lr, fp16=not use_bf16, bf16=use_bf16,
        logging_steps=1, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
        seed=42, output_dir=os.path.join(args.out, name + "_checkpoints"), report_to="none"))
    try:  # learn only from the assistant's answer, not from the long prompt
        from unsloth.chat_templates import train_on_responses_only
        trainer = train_on_responses_only(trainer, instruction_part=fmt["instruction_part"], response_part=fmt["response_part"])
    except Exception as e:
        print(f"      (train_on_responses_only unavailable, training on the full text instead: {e})")
    trainer.train()

    print("[4/5] Format check (a sanity check on a training prompt - NOT a quality test) ...")
    FastLanguageModel.for_inference(model)
    first = json.loads(open(data_path, encoding="utf-8").readline())["messages"]
    inputs = tokenizer(render_prompt(first[0]["content"], first[1]["content"], family), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=700, do_sample=True, temperature=0.2, use_cache=True)
    reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        json.loads(reply[reply.find("{"): reply.rfind("}") + 1])
        print("      model replied with valid JSON.")
    except ValueError:
        print("      WARNING: reply was not valid JSON. Raw reply:\n" + reply[:600])

    print("[5/5] Exporting for Ollama ...")
    os.makedirs(args.out, exist_ok=True)
    adapter_dir = os.path.join(args.out, name + "_lora")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    work = os.path.join(args.out, name + "_gguf_work")
    model.save_pretrained_gguf(work, tokenizer, quantization_method="q4_k_m")
    export_dir = os.path.join(args.out, name)
    os.makedirs(export_dir, exist_ok=True)
    shutil.copy(find_gguf(work), os.path.join(export_dir, "model.Q4_K_M.gguf"))
    with open(os.path.join(export_dir, "Modelfile"), "w", encoding="utf-8") as f:
        f.write(modelfile_text("model.Q4_K_M.gguf", family))
    print(f"\n[done] {export_dir}/ is ready. Copy the whole 'exports' folder to your computer, then run: python install_models.py")


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > sample_docs/cbdc_policy.txt << '__END_OF_FILE__'
CBDC - Existing Policy Notes (illustrative sample)

Wallets and limits: The central bank issues a retail digital currency through licensed intermediaries. Wallets are tiered by identity verification level, and each tier has a maximum balance and a daily transaction limit. Small payments should preserve user privacy, while larger payments must be traceable to authorised authorities.

Offline payments: The central bank wants citizens in low-connectivity regions to pay without a live network connection. The technology and risk model for offline value transfer are still being explored through pilots.

Interbank settlement: Intermediaries settle with the central bank through the existing real-time gross settlement (RTGS) system. The interface specification is fixed by the central bank, and any change goes through a formal approval process.

Programmable payments: Policymakers are studying conditional payments, such as time-limited subsidies. Operating targets are 99.99% availability and at least 2,000 transactions per second at peak.
__END_OF_FILE__
cat > sample_docs/cross_border_payments_policy.txt << '__END_OF_FILE__'
Cross-Border Payments - Existing Policy Notes (illustrative sample)

FX quotes: Customers must see a firm exchange-rate quote before they confirm a transfer. A quote is valid for 30 seconds; after that the customer must request a new one. The fee and the final amount received must be shown on the same screen as the rate.

Settlement: Value moves between the sending and receiving corridors over stablecoin rails. The sender's funds are converted to a USD stablecoin, transferred on-chain, and converted into local currency by a licensed partner in the receiving country. Target end-to-end settlement time is under 5 minutes.

Failures and refunds: If a transfer cannot complete, the sender must be refunded automatically within 24 hours and informed by notification. Operations staff reconcile on-chain transfers against partner bank ledgers every day, and any unmatched item is escalated the same day.

Compliance: Transfers above the corridor threshold require originator and beneficiary data to be exchanged with the receiving provider before settlement. Counterparties must be on the approved provider list.

Customers: Many remittance users have limited digital literacy and use low-end phones. The app must support several local languages, show a clear status for every transfer, and send SMS updates when connectivity is poor.
__END_OF_FILE__
cat > sample_docs/crypto_custody_policy.txt << '__END_OF_FILE__'
Crypto Custody - Existing Policy Notes (illustrative sample)

Storage: Client assets are held in segregated wallets. At least 95% of assets stay in cold storage; the hot wallet holds only the amount needed for expected daily withdrawals. Private keys are generated and kept inside hardware security modules, and signing uses multi-party computation so no single person or server holds a complete key.

Withdrawals: Withdrawals are sent only to whitelisted addresses. A newly added address has a 24-hour cooling period before it can receive funds, and the client is notified when an address is added. Withdrawals above a set value need approval from two authorised operators.

Signing service: The trading desk calls an internal signing API to move funds between wallets. The API must be available around the clock, log every request, and reject anything that violates policy limits.

Client app: Retail clients use a mobile wallet app to view balances, receive funds and request withdrawals. The business expects frequent changes to its screens and features as it learns from user feedback.
__END_OF_FILE__
cat > sample_docs/defi_lending_policy.txt << '__END_OF_FILE__'
DeFi Lending Platform - Existing Policy Notes (illustrative sample)

Withdrawals: Standard withdrawals below 10,000 USDC equivalent are processed instantly by the smart contract once collateral health checks pass. Large withdrawals above this threshold have historically been a source of incidents when a single compromised key authorized a transfer. The business wants additional friction for high-value transfers without harming normal users.

Collateralization: Borrowers must maintain a minimum collateralization ratio. If the ratio falls below the liquidation threshold, the protocol liquidates part of the position. Oracle price feeds drive this calculation, so any delay or manipulation of the price feed directly affects user funds.

Interest rates: Rates float based on pool utilization and are recalculated on every block. Rate changes must be visible to users before they confirm a transaction so they are not surprised by slippage.

Auditability: All lending pool parameter changes (collateral factors, liquidation thresholds, interest rate curves) must be logged with the proposer identity and timestamp for later audit.

Users: Lenders and borrowers want to see their positions, health factor and accrued interest in the app. The interface is expected to evolve as the team gathers feedback.
__END_OF_FILE__
cat > sample_docs/kyc_aml_policy.txt << '__END_OF_FILE__'
Blockchain KYC / AML - Existing Policy Notes (illustrative sample)

Customer onboarding: New users must complete identity verification before any wallet linked to their account can send funds off-platform. Verification tiers determine daily and monthly transfer limits.

Travel Rule: When a transaction moves value between two virtual-asset service providers above the jurisdictional threshold, originator and beneficiary information must be collected and shared with the counterparty institution before the transfer settles.

Sanctions screening: Wallet addresses are screened against sanctions lists at onboarding and periodically thereafter. A positive match freezes outbound transfers pending manual compliance review.

Record retention: Identity verification records and transaction monitoring alerts must be retained for the period required by local regulation and made available to auditors and regulators on request.

Suspicious activity reporting: Confirmed suspicious activity must be reported to the regulator within the deadline set by law. The report format and review workflow are prescribed by the regulator and change rarely; every report needs sign-off from the compliance officer.

Reusable identity: The business is evaluating reusable digital identity credentials issued by trusted providers so customers need not repeat verification at each service. Standards and regulatory acceptance are still evolving.
__END_OF_FILE__
cat > sample_docs/stablecoin_reserve_policy.txt << '__END_OF_FILE__'
Stablecoin Issuance - Existing Policy Notes (illustrative sample)

Reserves: Every token in circulation must be backed 1:1 by cash and short-term government securities held with segregated, regulated custodians. New tokens may be minted only after the matching reserve deposit has been confirmed by two independent officers.

Redemption: Holders can redeem at par. Redemption requests are honoured within one business day, and the corresponding tokens are burned on-chain before fiat is released.

Attestation: An independent firm attests to reserve balances every month, and the summary is published to token holders and the regulator within five business days of month end. The format and content of the report are fixed by the regulator.

Market stress: If the market price stays more than 1% away from the peg for 15 minutes, minting must pause and the risk committee must be alerted. The business is also researching an algorithmic supply-adjustment mechanism to supplement reserves; its behaviour under stress is not yet well understood.
__END_OF_FILE__
cat > sample_docs/tokenized_trade_finance_policy.txt << '__END_OF_FILE__'
Tokenized Trade Finance - Existing Policy Notes (illustrative sample)

Letters of credit: Letters of credit are issued between a small group of consortium banks on a permissioned ledger. The terms follow established international documentary-credit rules and have been stable for years. Any change to the process needs sign-off from every member bank and the consortium's legal team.

Documents: Trade documents such as invoices and bills of lading are hashed and anchored on the ledger so that later tampering can be detected. Shipment status arrives from logistics partners through external data feeds whose formats and reliability vary.

Invoice financing: Exporters can tokenize approved invoices so that several investors can each fund part of an invoice. Investors are repaid when the importer pays. The product is new, and the business expects to adjust its features after pilot feedback.

Settlement and disputes: Payment is settled between banks once documents are accepted. Disputes over discrepancies are handled through a defined review process and every decision must be recorded.
__END_OF_FILE__
echo "Created $(ls | wc -l) items:"; ls

### 3. Check that everything was created

In [ ]:
import os
need = ["finetune_llm.py", "generate_dataset.py", "pipeline.py", "retriever.py", "llm_clients.py", "seed_data.py"]
missing = [f for f in need if not os.path.exists(f)]
docs = os.listdir("sample_docs") if os.path.isdir("sample_docs") else []
assert not missing and docs, f"Missing: {missing or 'sample_docs/*'}. Re-run step 2 and read its output for errors."
print("OK - files present:", sorted(os.listdir(".")))
print("Sample documents:", docs)

### 4. (Optional) Use your own edited training data
Only if you edited `seed_data.py` on your computer and did **not** rebuild this notebook with `python make_notebook.py`.
Remove the `#` at the start of the three lines, run the cell, and choose your `seed_data.py`. Otherwise skip this cell.

In [ ]:
# from google.colab import files
# up = files.upload()                     # choose your seed_data.py
# import shutil; shutil.move(list(up)[0], "seed_data.py")

### 5. Install Unsloth (takes a few minutes; if Colab asks you to restart the runtime, do it, then continue from step 6 - your files stay)

In [ ]:
!pip install -q unsloth

### 6. Build the training data (28 examples per model, from `seed_data.py`)

In [ ]:
!python generate_dataset.py

### 7. Fine-tune LLM 1 (Llama-3.2-3B -> requirements). Roughly 10-30 minutes including the GGUF export.

In [ ]:
!python finetune_llm.py --stage 1
!rm -rf exports/blockchain-req-llm_gguf_work exports/blockchain-req-llm_checkpoints   # free disk space

### 8. Fine-tune LLM 2 (Qwen2.5-3B -> SDLC only)

In [ ]:
!python finetune_llm.py --stage 2
!rm -rf exports/blockchain-sdlc-llm_gguf_work exports/blockchain-sdlc-llm_checkpoints

### 9. Get the two models out of Colab
**Do this before the session ends: Colab deletes its files afterwards. Do not restart or disconnect the runtime until you have saved them.**
First check that training produced the files (you should see two folders, each with a `Modelfile` and a ~2 GB `model.Q4_K_M.gguf`):

In [ ]:
!ls -lh exports/blockchain-req-llm exports/blockchain-sdlc-llm

**Option A - Google Drive (recommended).** A pop-up asks you to connect Google Drive: allow pop-ups for this site, pick the **same Google account** that Colab is signed in with, and click *Allow* / *Continue* on every screen.
If you get `mount failed`, run the cell again, or use Option B below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/drive/MyDrive/blockchain_llms
!cp -r exports/blockchain-req-llm exports/blockchain-sdlc-llm /content/drive/MyDrive/blockchain_llms/
!ls -lh /content/drive/MyDrive/blockchain_llms/*

**Option B - download directly (use this if Drive keeps failing).** Makes one zip per model (~2 GB each) and downloads them.
On your computer, create a folder named `exports` inside your project folder and unzip **both** zips into it.

In [ ]:
import os, subprocess
from google.colab import files
for name in ["blockchain-req-llm", "blockchain-sdlc-llm"]:
    subprocess.run(["zip", "-0", "-qr", f"../{name}.zip", name], cwd="exports", check=True)
    print(name, round(os.path.getsize(name + ".zip") / 1e9, 2), "GB - downloading...")
    files.download(name + ".zip")

### 10. (Optional) Download the complete project code
Only needed if you don't already have the project folder on your computer. It builds one small zip with every project file.

In [ ]:
%%bash
set -e
mkdir -p sample_docs
cat > web_app.py << '__END_OF_FILE__'
"""Local website for the pipeline.

    python web_app.py        ->  opens http://localhost:8000

The page runs the three steps one after another (so slow CPU models don't hit a browser timeout,
and each result appears as soon as it is ready):
    POST /api/retrieve      Step 1  RAG retrieval
    POST /api/requirements  Step 2  LLM 1 -> requirements (+ grounding check)
    POST /api/sdlc          Step 3  LLM 2 -> SDLC model only
    GET  /api/config        which Ollama models are installed + sensible defaults
"""
from __future__ import annotations

import json
import os
import threading
import traceback
import webbrowser
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

from llm_clients import (DEFAULT_GEMINI_MODEL, LLMError, build_client, ollama_installed_models,
                         pick_default_ollama_model)
from pipeline import (MIN_GROUNDING, build_index, generate_requirements, ground_requirements,
                      identify_sdlc, load_documents, retrieve_evidence)

BASE_DIR = Path(__file__).resolve().parent
PORT = int(os.environ.get("PORT", "8000"))

HTML_PAGE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8" />
<meta name="viewport" content="width=device-width, initial-scale=1.0" />
<title>Blockchain Finance: Requirements &amp; SDLC Pipeline</title>
<style>
  :root { --bg:#0f172a; --card:#1e293b; --border:#334155; --text:#f8fafc; --muted:#94a3b8; --accent:#38bdf8;
          --ok:#22c55e; --warn:#f59e0b; --bad:#f87171; }
  * { box-sizing:border-box; font-family:system-ui,-apple-system,"Segoe UI",sans-serif; }
  body { margin:0; padding:28px 16px 60px; background:var(--bg); color:var(--text); }
  .container { max-width:960px; margin:0 auto; }
  h1 { margin:0 0 6px; font-size:25px; } h2 { margin:0 0 12px; font-size:18px; }
  p.sub { margin:0 0 18px; color:var(--muted); font-size:14px; }
  .card { background:var(--card); border:1px solid var(--border); border-radius:10px; padding:18px; margin-bottom:16px; }
  .grid { display:grid; grid-template-columns:1fr 1fr; gap:14px; margin-bottom:14px; }
  @media (max-width:700px){ .grid{ grid-template-columns:1fr; } }
  label { display:block; font-size:12.5px; color:var(--muted); margin-bottom:5px; font-weight:600; }
  input, select, textarea { width:100%; padding:9px 11px; border-radius:6px; border:1px solid var(--border); background:#0b1120; color:var(--text); font-size:14px; }
  textarea { resize:vertical; min-height:64px; }
  details { margin-bottom:14px; } summary { cursor:pointer; color:var(--muted); font-size:13px; margin-bottom:8px; }
  button { background:var(--accent); color:#0f172a; border:0; padding:12px 18px; border-radius:6px; font-weight:700; font-size:15px; cursor:pointer; width:100%; }
  button:disabled { opacity:.5; cursor:wait; }
  .status { font-size:13px; line-height:1.6; margin-bottom:16px; padding:12px 14px; border-radius:8px; border:1px solid var(--border); background:#0b1120; }
  .status .ok { color:var(--ok); } .status .warn { color:var(--warn); } .status .bad { color:var(--bad); }
  .stepper { display:flex; gap:8px; margin:0 0 16px; flex-wrap:wrap; }
  .pill { flex:1; min-width:150px; padding:8px 12px; border-radius:20px; border:1px solid var(--border); font-size:12.5px; color:var(--muted); text-align:center; }
  .pill.active { border-color:var(--accent); color:var(--accent); } .pill.done { border-color:var(--ok); color:var(--ok); } .pill.error { border-color:var(--bad); color:var(--bad); }
  .item { padding:11px 12px; border:1px solid var(--border); border-radius:8px; margin-bottom:9px; background:#0b1120; font-size:14px; }
  .item.review { border-color:var(--warn); }
  .badge { display:inline-block; padding:2px 8px; border-radius:4px; font-size:11.5px; font-weight:700; background:#0284c7; color:#fff; margin-right:6px; }
  .badge.alt { background:#475569; } .badge.src { background:#0d9488; }
  .meta { margin-top:6px; font-size:12.5px; color:var(--muted); } .meta.warn { color:var(--warn); }
  .sdlc { font-size:22px; font-weight:800; color:var(--ok); margin:4px 0 10px; }
  .error { display:none; color:var(--bad); background:#450a0a; border:1px solid #ef4444; padding:12px; border-radius:8px; margin-bottom:16px; font-size:14px; white-space:pre-wrap; }
  #results { display:none; }
</style>
</head>
<body>
<div class="container">
  <h1>Blockchain Finance: Requirements &amp; SDLC Pipeline</h1>
  <p class="sub">Input &rarr; <b>RAG retrieval</b> &rarr; <b>LLM 1</b> (requirements + reasoning) &rarr; <b>LLM 2</b> (SDLC model only) &rarr; final output</p>

  <div class="status" id="status">Checking Ollama&hellip;</div>

  <div class="card">
    <div class="grid">
      <div><label>Domain</label><input id="domain" value="DeFi Lending" /></div>
      <div><label>Document folder (RAG corpus: .txt / .md files)</label><input id="docs" value="sample_docs" /></div>
    </div>
    <div style="margin-bottom:14px"><label>Functionality to generate requirements for</label>
      <textarea id="functionality">Multi-signature approval for large withdrawals from a lending pool</textarea></div>
    <details><summary>Optional: paste extra document text to add to the corpus</summary>
      <textarea id="extra" placeholder="Paste an existing requirement document or policy here..."></textarea></details>
    <div class="grid">
      <div><label>LLM 1 provider (requirements)</label>
        <select id="p1" onchange="providerChanged(1)"><option value="ollama">Ollama (local)</option><option value="gemini">Gemini (cloud)</option></select></div>
      <div><label>LLM 1 model</label><input id="m1" list="modelList" /></div>
    </div>
    <div class="grid">
      <div><label>LLM 2 provider (SDLC only)</label>
        <select id="p2" onchange="providerChanged(2)"><option value="ollama">Ollama (local)</option><option value="gemini">Gemini (cloud)</option></select></div>
      <div><label>LLM 2 model</label><input id="m2" list="modelList" /></div>
    </div>
    <datalist id="modelList"></datalist>
    <div class="grid">
      <div id="keyBox" style="display:none"><label>Gemini API key (kept in memory, sent only to this local server)</label><input type="password" id="apiKey" placeholder="AIza..." /></div>
      <div><label>Evidence chunks to retrieve (top-k)</label><input type="number" id="topk" value="4" min="1" max="8" /></div>
    </div>
    <button id="runBtn" onclick="runPipeline()">Run pipeline</button>
  </div>

  <div class="error" id="errorBox"></div>
  <div class="stepper">
    <div class="pill" id="s1">1 &middot; RAG retrieval</div>
    <div class="pill" id="s2">2 &middot; LLM 1: requirements</div>
    <div class="pill" id="s3">3 &middot; LLM 2: SDLC</div>
  </div>

  <div id="results">
    <div class="card" id="c1" style="display:none"><h2>Step 1 &mdash; Evidence retrieved from your documents</h2><div id="evidence"></div></div>
    <div class="card" id="c2" style="display:none"><h2>Step 2 &mdash; Requirements <span id="m1used" class="meta"></span></h2>
      <div id="reqs"></div><p class="meta"><b>Reasoning:</b> <span id="reasoning"></span></p></div>
    <div class="card" id="c3" style="display:none"><h2>Step 3 &mdash; Recommended SDLC <span id="m2used" class="meta"></span></h2>
      <div class="sdlc" id="sdlcName"></div><label>Key factors</label><ul id="factors"></ul>
      <p style="font-size:14px;line-height:1.55"><b>Software-engineering justification:</b> <span id="justification"></span></p></div>
  </div>
</div>

<script>
const $ = id => document.getElementById(id);
const esc = s => String(s ?? '').replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
let CFG = null;

async function api(path, body) {
  const r = await fetch(path, {method:'POST', headers:{'Content-Type':'application/json'}, body: JSON.stringify(body)});
  const d = await r.json().catch(() => ({error:'The server sent an unreadable response.'}));
  if (!r.ok || d.error) throw new Error(d.error || 'Request failed');
  return d;
}

function setDefaultModel(step) {
  const prov = $('p' + step).value;
  $('m' + step).value = prov === 'gemini' ? CFG.gemini_model : CFG.defaults['llm' + step].model;
}
function providerChanged(step) {
  setDefaultModel(step);
  $('keyBox').style.display = ($('p1').value === 'gemini' || $('p2').value === 'gemini') && !CFG.gemini_key_in_env ? 'block' : 'none';
}

async function loadConfig() {
  const r = await fetch('/api/config'); CFG = await r.json();
  setDefaultModel(1); setDefaultModel(2);
  $('modelList').innerHTML = CFG.installed.map(n => `<option value="${esc(n)}">`).join('');
  const lines = [];
  if (!CFG.ollama_up) {
    lines.push('<span class="bad">Ollama is not running.</span> Start the Ollama app (or run <code>ollama serve</code>), then reload this page &mdash; or switch a step to Gemini.');
  } else {
    lines.push('<span class="ok">Ollama is running.</span> Installed models: ' + (CFG.installed.length ? esc(CFG.installed.join(', ')) : 'none'));
    [1, 2].forEach(n => {
      const d = CFG.defaults['llm' + n], have = CFG.installed.some(x => x.split(':')[0] === d.model.split(':')[0]);
      if (d.finetuned) lines.push(`LLM ${n}: <span class="ok">using your fine-tuned model <b>${esc(d.model)}</b></span>`);
      else if (have) lines.push(`LLM ${n}: fine-tuned model not installed &mdash; using base model <b>${esc(d.model)}</b> (works fine; fine-tuning is optional, see README)`);
      else lines.push(`LLM ${n}: <span class="warn">base model <b>${esc(d.model)}</b> is not installed &mdash; run <code>ollama pull ${esc(d.model)}</code></span>`);
    });
  }
  $('status').innerHTML = lines.join('<br>');
}

function setStep(n, state) { $('s' + n).className = 'pill ' + state; }

async function runPipeline() {
  const btn = $('runBtn'), err = $('errorBox');
  btn.disabled = true; err.style.display = 'none'; $('results').style.display = 'none';
  ['c1','c2','c3'].forEach(id => $(id).style.display = 'none'); [1,2,3].forEach(n => setStep(n, ''));
  let current = 1;
  const common = {docs: $('docs').value.trim(), extra_text: $('extra').value, domain: $('domain').value.trim(),
                  functionality: $('functionality').value.trim(), top_k: parseInt($('topk').value || '4', 10)};
  const key = $('apiKey').value.trim();
  try {
    $('results').style.display = 'block';
    setStep(1, 'active'); btn.textContent = 'Step 1/3: retrieving evidence...';
    const r1 = await api('/api/retrieve', common);
    $('evidence').innerHTML = r1.evidence.length ? r1.evidence.map(e =>
      `<div class="item"><span class="badge">${esc(e.id)}</span><span class="badge src">${esc(e.source)}</span><span class="meta">score ${esc(e.score)}</span>
       <div style="margin-top:6px;font-size:13px;color:#cbd5e1">${esc(e.text)}</div></div>`).join('')
      : '<div class="meta warn">No relevant evidence was found in your documents. The model will rely on general domain knowledge and every requirement will be flagged for review.</div>';
    $('c1').style.display = 'block'; setStep(1, 'done');

    current = 2; setStep(2, 'active'); btn.textContent = 'Step 2/3: LLM 1 is writing requirements (can take a minute on CPU)...';
    const r2 = await api('/api/requirements', {...common, evidence: r1.evidence, provider: $('p1').value, model: $('m1').value.trim(), api_key: key});
    $('reqs').innerHTML = r2.requirements.map(r => `<div class="item ${r.needs_review ? 'review' : ''}">
      <span class="badge">${esc(r.id)}</span><span class="badge alt">${esc(r.category)}</span>
      <div style="margin-top:6px">${esc(r.statement)}</div>
      <div class="meta ${r.needs_review ? 'warn' : ''}">${r.needs_review ? '&#9888; Needs human review &mdash; weakly supported by the retrieved evidence' : 'Supported by the evidence'}
        (closest: ${esc(r.support_source || 'none')}, similarity ${esc(r.support_score)})</div></div>`).join('');
    $('reasoning').textContent = r2.reasoning; $('m1used').textContent = '(' + r2.model + ')';
    $('c2').style.display = 'block'; setStep(2, 'done');

    current = 3; setStep(3, 'active'); btn.textContent = 'Step 3/3: LLM 2 is choosing the SDLC model...';
    const r3 = await api('/api/sdlc', {domain: common.domain, evidence: r1.evidence, requirements: r2.requirements,
                                       provider: $('p2').value, model: $('m2').value.trim(), api_key: key});
    $('sdlcName').textContent = r3.sdlc.recommended_sdlc; $('m2used').textContent = '(' + r3.model + ')';
    $('factors').innerHTML = r3.sdlc.key_factors.map(f => `<li>${esc(f)}</li>`).join('');
    $('justification').textContent = r3.sdlc.justification;
    $('c3').style.display = 'block'; setStep(3, 'done');
  } catch (e) {
    setStep(current, 'error'); err.textContent = 'Error: ' + e.message; err.style.display = 'block';
  } finally { btn.disabled = false; btn.textContent = 'Run pipeline'; }
}
loadConfig();
</script>
</body>
</html>
"""


# ----------------------------------------------------------------------------- API helpers
def _text(body: dict, key: str, required: bool = False) -> str:
    value = str(body.get(key) or "").strip()
    if required and not value:
        raise ValueError(f"'{key}' is required.")
    return value


def _retriever(body: dict):
    folder = _text(body, "docs")
    if folder:
        path = Path(folder).expanduser()
        folder = str(path if path.is_absolute() else BASE_DIR / path)
    return build_index(load_documents(folder, body.get("extra_text") or ""))


def _list_of_dicts(body: dict, key: str, fields: tuple) -> list:
    items = body.get(key)
    if not isinstance(items, list) or any(not isinstance(i, dict) for i in items):
        raise ValueError(f"'{key}' must be a list.")
    for i in items:
        for f in fields:
            i.setdefault(f, "")
    return items


def api_config() -> dict:
    installed = ollama_installed_models()
    return {
        "ollama_up": installed is not None,
        "installed": installed or [],
        "defaults": {"llm1": pick_default_ollama_model("llm1", installed), "llm2": pick_default_ollama_model("llm2", installed)},
        "gemini_model": DEFAULT_GEMINI_MODEL,
        "gemini_key_in_env": bool(os.environ.get("GEMINI_API_KEY")),
        "min_grounding": MIN_GROUNDING,
    }


def api_retrieve(body: dict) -> dict:
    top_k = max(1, min(int(body.get("top_k") or 4), 8))
    return {"evidence": retrieve_evidence(_retriever(body), _text(body, "domain", True), _text(body, "functionality", True), top_k)}


def api_requirements(body: dict) -> dict:
    domain, functionality = _text(body, "domain", True), _text(body, "functionality", True)
    evidence = _list_of_dicts(body, "evidence", ("id", "source", "text"))
    llm1 = build_client(_text(body, "provider") or "ollama", _text(body, "model") or None, _text(body, "api_key") or None, role="llm1")
    draft = generate_requirements(domain, functionality, evidence, llm1)
    reqs = ground_requirements(_retriever(body), draft["requirements"], evidence)
    return {"requirements": reqs, "reasoning": draft["reasoning"], "model": llm1.label,
            "needs_review_count": sum(r["needs_review"] for r in reqs)}


def api_sdlc(body: dict) -> dict:
    domain = _text(body, "domain", True)
    evidence = _list_of_dicts(body, "evidence", ("id", "source", "text"))
    reqs = _list_of_dicts(body, "requirements", ("id", "statement", "category"))
    llm2 = build_client(_text(body, "provider") or "ollama", _text(body, "model") or None, _text(body, "api_key") or None, role="llm2")
    return {"sdlc": identify_sdlc(domain, reqs, evidence, llm2), "model": llm2.label}


ROUTES = {"/api/retrieve": api_retrieve, "/api/requirements": api_requirements, "/api/sdlc": api_sdlc}


class Handler(BaseHTTPRequestHandler):
    def _send(self, code: int, payload, content_type: str = "application/json") -> None:
        data = payload.encode("utf-8") if isinstance(payload, str) else json.dumps(payload).encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", content_type + "; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path in ("/", "/index.html"):
            self._send(200, HTML_PAGE, "text/html")
        elif self.path == "/api/config":
            self._send(200, api_config())
        else:
            self._send(404, {"error": "Not found"})

    def do_POST(self):
        handler = ROUTES.get(self.path)
        if not handler:
            return self._send(404, {"error": "Not found"})
        try:
            body = json.loads(self.rfile.read(int(self.headers.get("Content-Length") or 0)).decode("utf-8") or "{}")
            self._send(200, handler(body))
        except (ValueError, LLMError) as e:       # user / model problems: show the message in the page
            self._send(400, {"error": str(e)})
        except Exception as e:                    # genuine bugs: log full trace in the terminal
            traceback.print_exc()
            self._send(500, {"error": f"Unexpected error: {e}"})

    def log_message(self, fmt, *args):            # keep the terminal readable (and never echo request bodies)
        pass


def main() -> None:
    url = f"http://localhost:{PORT}"
    try:
        server = ThreadingHTTPServer(("127.0.0.1", PORT), Handler)
    except OSError as e:
        raise SystemExit(f"Cannot start on port {PORT} ({e}). Close whatever is using it, or set PORT=8001.")
    print(f"\n[ok] Website running at {url}   (Ctrl+C to stop)")
    threading.Timer(0.8, webbrowser.open, [url]).start()
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        print("\nStopped.")


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > main.py << '__END_OF_FILE__'
"""Command-line version of the pipeline.

    python main.py --domain "DeFi Lending" --functionality "Multi-signature approval for large withdrawals"

By default both steps run on local Ollama models (two different ones):
  LLM 1 -> blockchain-req-llm (fine-tuned) if installed, otherwise llama3.2:3b
  LLM 2 -> blockchain-sdlc-llm (fine-tuned) if installed, otherwise qwen2.5:3b
"""
from __future__ import annotations

import argparse
import json
import os
import sys

from llm_clients import LLMError, build_client
from pipeline import MIN_GROUNDING, build_index, load_documents, run_pipeline

HERE = os.path.dirname(os.path.abspath(__file__))


def main() -> None:
    p = argparse.ArgumentParser(description="RAG -> LLM 1 (requirements) -> LLM 2 (SDLC only)")
    p.add_argument("--docs", default=os.path.join(HERE, "sample_docs"), help="Folder of existing requirement documents (.txt/.md)")
    p.add_argument("--domain", required=True, help='e.g. "DeFi Lending"')
    p.add_argument("--functionality", required=True, help="Feature to generate requirements for")
    p.add_argument("--llm1-provider", default="ollama", choices=["ollama", "gemini"])
    p.add_argument("--llm1-model", default=None, help="default: blockchain-req-llm if installed, else llama3.2:3b")
    p.add_argument("--llm2-provider", default="ollama", choices=["ollama", "gemini"])
    p.add_argument("--llm2-model", default=None, help="default: blockchain-sdlc-llm if installed, else qwen2.5:3b")
    p.add_argument("--top-k", type=int, default=4, help="Chunks retrieved by RAG")
    p.add_argument("--min-grounding", type=float, default=MIN_GROUNDING, help="Below this support score a requirement is flagged for review")
    p.add_argument("--json-out", default=None, help="Also save the full result to this JSON file")
    args = p.parse_args()

    try:
        retriever = build_index(load_documents(args.docs))
        llm1 = build_client(args.llm1_provider, args.llm1_model, role="llm1")
        llm2 = build_client(args.llm2_provider, args.llm2_model, role="llm2")
        result = run_pipeline(retriever, args.domain, args.functionality, llm1, llm2,
                              top_k=args.top_k, min_grounding=args.min_grounding, progress=lambda m: print(f"\n>> {m}"))
    except (LLMError, ValueError) as e:
        sys.exit(f"\nError: {e}")

    print("\n=== Step 1: RAG evidence ===")
    for e in result["evidence"] or [{"id": "-", "source": "(none)", "text": "", "score": 0}]:
        print(f"[{e['id']}] {e['source']} (score {e['score']}): {e['text'][:110]}...")
    print(f"\n=== Step 2: Requirements from {result['models']['llm1']} ===")
    for r in result["requirements"]:
        flag = "  <-- NEEDS HUMAN REVIEW (weakly supported by evidence)" if r["needs_review"] else ""
        print(f"{r['id']} [{r['category']}] {r['statement']}\n      closest evidence: {r['support_source']} (similarity {r['support_score']}){flag}")
    print("\nReasoning:", result["requirements_reasoning"])
    s = result["sdlc"]
    print(f"\n=== Step 3: SDLC from {result['models']['llm2']} ===\nRecommended: {s['recommended_sdlc']}")
    for f in s["key_factors"]:
        print(f"  - {f}")
    print("Justification:", s["justification"])

    if args.json_out:
        with open(args.json_out, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2, ensure_ascii=False)
        print(f"\nSaved full result to {args.json_out}")


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > install_models.py << '__END_OF_FILE__'
"""Register your fine-tuned models in Ollama (run this on the computer where Ollama is installed).

    python install_models.py                 # looks in ./exports
    python install_models.py --exports D:\\downloads\\exports

For every <exports>/<name>/Modelfile it runs `ollama create <name> -f <that Modelfile>`.
After that, the website and CLI pick the fine-tuned models up automatically.
"""
from __future__ import annotations

import argparse
import os
import shutil
import subprocess
import sys


def main() -> None:
    p = argparse.ArgumentParser(description="Create Ollama models from fine-tuned exports")
    p.add_argument("--exports", default="exports", help="folder containing blockchain-req-llm/ and blockchain-sdlc-llm/")
    args = p.parse_args()

    if not shutil.which("ollama"):
        sys.exit("`ollama` was not found on PATH. Install it from https://ollama.com and make sure the app is running.")
    if not os.path.isdir(args.exports):
        sys.exit(f"Folder not found: {args.exports}")

    found = 0
    for name in sorted(os.listdir(args.exports)):
        modelfile = os.path.join(args.exports, name, "Modelfile")
        gguf = os.path.join(args.exports, name, "model.Q4_K_M.gguf")
        if not os.path.isfile(modelfile):
            continue
        if not os.path.isfile(gguf):
            print(f"[skip] {name}: model.Q4_K_M.gguf is missing next to the Modelfile")
            continue
        found += 1
        print(f"\n[*] ollama create {name} ...")
        if subprocess.call(["ollama", "create", name, "-f", modelfile]) != 0:
            sys.exit(f"`ollama create {name}` failed. Is the Ollama app running?")
    if not found:
        sys.exit(f"No <name>/Modelfile found inside {args.exports}.")
    print("\n[ok] Done. Installed models:")
    subprocess.call(["ollama", "list"])


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > make_notebook.py << '__END_OF_FILE__'
"""Build finetune_colab.ipynb with the CURRENT project files embedded inside it.

    python make_notebook.py

Why: the notebook then needs no zip upload - Colab creates every file itself from the notebook, so there is
nothing to zip wrongly. If you edit seed_data.py (to add training examples) or any other file, re-run this
script and upload the new notebook to Colab.
"""
from __future__ import annotations

import json
import os

HERE = os.path.dirname(os.path.abspath(__file__))
DELIM = "__END_OF_FILE__"

# needed for training
TRAIN_FILES = ["retriever.py", "llm_clients.py", "pipeline.py", "seed_data.py", "generate_dataset.py", "finetune_llm.py"]
# the rest of the project (optional: lets Colab hand you the complete project as one zip)
EXTRA_FILES = ["web_app.py", "main.py", "install_models.py", "make_notebook.py", "test_pipeline.py", "requirements.txt", "README.md"]


def _read(rel: str) -> str:
    with open(os.path.join(HERE, rel), encoding="utf-8") as f:
        text = f.read().replace("\r\n", "\n").rstrip("\n") + "\n"
    if any(line == DELIM for line in text.split("\n")):     # only a line that is exactly the delimiter would end the heredoc
        raise ValueError(f"{rel} contains a line equal to the reserved delimiter {DELIM}")
    return text


def _doc_files() -> list:
    return [os.path.join("sample_docs", n) for n in sorted(os.listdir(os.path.join(HERE, "sample_docs")))
            if n.lower().endswith((".txt", ".md"))]


def bash_cell(files: list) -> str:
    """One `%%bash` cell that writes every file with a quoted heredoc."""
    lines = ["%%bash", "set -e", "mkdir -p sample_docs"]
    for rel in files:
        lines += [f"cat > {rel} << '{DELIM}'", _read(rel).rstrip("\n"), DELIM]
    lines.append('echo "Created $(ls | wc -l) items:"; ls')
    return "\n".join(lines)


def _md(text: str) -> dict:
    return {"cell_type": "markdown", "metadata": {}, "source": text.splitlines(keepends=True)}


def _code(text: str) -> dict:
    return {"cell_type": "code", "metadata": {}, "execution_count": None, "outputs": [], "source": text.splitlines(keepends=True)}


def build() -> dict:
    cells = [
        _md("""# Fine-tune the two models (free Colab GPU)

**You do not need to upload or zip anything.** This notebook already contains all the project code and creates the files itself.

**Before you start:** menu *Runtime -> Change runtime type -> **T4 GPU** -> Save*. Then run the cells below **one by one, in order** (click the play button on each, wait for it to finish)."""),
        _md("### 1. Check that a GPU is attached (you should see a table that mentions a Tesla T4)"),
        _code("!nvidia-smi"),
        _md("### 2. Create the project files inside Colab (just run it - there is no need to read the code)"),
        _code(bash_cell(TRAIN_FILES + _doc_files())),
        _md("### 3. Check that everything was created"),
        _code('''import os
need = ["finetune_llm.py", "generate_dataset.py", "pipeline.py", "retriever.py", "llm_clients.py", "seed_data.py"]
missing = [f for f in need if not os.path.exists(f)]
docs = os.listdir("sample_docs") if os.path.isdir("sample_docs") else []
assert not missing and docs, f"Missing: {missing or 'sample_docs/*'}. Re-run step 2 and read its output for errors."
print("OK - files present:", sorted(os.listdir(".")))
print("Sample documents:", docs)'''),
        _md("""### 4. (Optional) Use your own edited training data
Only if you edited `seed_data.py` on your computer and did **not** rebuild this notebook with `python make_notebook.py`.
Remove the `#` at the start of the three lines, run the cell, and choose your `seed_data.py`. Otherwise skip this cell."""),
        _code('''# from google.colab import files
# up = files.upload()                     # choose your seed_data.py
# import shutil; shutil.move(list(up)[0], "seed_data.py")'''),
        _md("### 5. Install Unsloth (takes a few minutes; if Colab asks you to restart the runtime, do it, then continue from step 6 - your files stay)"),
        _code("!pip install -q unsloth"),
        _md("### 6. Build the training data (28 examples per model, from `seed_data.py`)"),
        _code("!python generate_dataset.py"),
        _md("### 7. Fine-tune LLM 1 (Llama-3.2-3B -> requirements). Roughly 10-30 minutes including the GGUF export."),
        _code("""!python finetune_llm.py --stage 1
!rm -rf exports/blockchain-req-llm_gguf_work exports/blockchain-req-llm_checkpoints   # free disk space"""),
        _md("### 8. Fine-tune LLM 2 (Qwen2.5-3B -> SDLC only)"),
        _code("""!python finetune_llm.py --stage 2
!rm -rf exports/blockchain-sdlc-llm_gguf_work exports/blockchain-sdlc-llm_checkpoints"""),
        _md("""### 9. Get the two models out of Colab
**Do this before the session ends: Colab deletes its files afterwards. Do not restart or disconnect the runtime until you have saved them.**
First check that training produced the files (you should see two folders, each with a `Modelfile` and a ~2 GB `model.Q4_K_M.gguf`):"""),
        _code("!ls -lh exports/blockchain-req-llm exports/blockchain-sdlc-llm"),
        _md("""**Option A - Google Drive (recommended).** A pop-up asks you to connect Google Drive: allow pop-ups for this site, pick the **same Google account** that Colab is signed in with, and click *Allow* / *Continue* on every screen.
If you get `mount failed`, run the cell again, or use Option B below."""),
        _code("""from google.colab import drive
drive.mount('/content/drive', force_remount=True)
!mkdir -p /content/drive/MyDrive/blockchain_llms
!cp -r exports/blockchain-req-llm exports/blockchain-sdlc-llm /content/drive/MyDrive/blockchain_llms/
!ls -lh /content/drive/MyDrive/blockchain_llms/*"""),
        _md("""**Option B - download directly (use this if Drive keeps failing).** Makes one zip per model (~2 GB each) and downloads them.
On your computer, create a folder named `exports` inside your project folder and unzip **both** zips into it."""),
        _code("""import os, subprocess
from google.colab import files
for name in ["blockchain-req-llm", "blockchain-sdlc-llm"]:
    subprocess.run(["zip", "-0", "-qr", f"../{name}.zip", name], cwd="exports", check=True)
    print(name, round(os.path.getsize(name + ".zip") / 1e9, 2), "GB - downloading...")
    files.download(name + ".zip")"""),
        _md("""### 10. (Optional) Download the complete project code
Only needed if you don't already have the project folder on your computer. It builds one small zip with every project file."""),
        _code(bash_cell(EXTRA_FILES)),
        _code("""!zip -qr blockchain_rag_project_code.zip *.py *.md requirements.txt sample_docs train_stage1.jsonl train_stage2.jsonl
from google.colab import files
files.download("blockchain_rag_project_code.zip")"""),
        _md("""### 11. On your own computer
1. Put the models in a folder named `exports` inside your project folder, so you have `exports/blockchain-req-llm/` and
   `exports/blockchain-sdlc-llm/` (each with a `Modelfile` and `model.Q4_K_M.gguf`). Option A: download the `blockchain_llms`
   folder from Google Drive and rename it `exports`. Option B: unzip both downloaded zips into `exports`.
2. Make sure the Ollama app is running, then in the project folder run: `python install_models.py`
3. Start the website: `python web_app.py` - the status box will say it is using your fine-tuned models."""),
    ]
    return {"cells": cells, "nbformat": 4, "nbformat_minor": 4, "metadata": {
        "accelerator": "GPU", "colab": {"name": "finetune_colab.ipynb", "provenance": []},
        "kernelspec": {"name": "python3", "display_name": "Python 3"}, "language_info": {"name": "python"}}}


def main() -> None:
    out = os.path.join(HERE, "finetune_colab.ipynb")
    with open(out, "w", encoding="utf-8") as f:
        json.dump(build(), f, indent=1, ensure_ascii=False)
    print(f"[ok] wrote {out} ({os.path.getsize(out) // 1024} KB) with {len(TRAIN_FILES) + len(EXTRA_FILES)} project files + sample_docs embedded")


if __name__ == "__main__":
    main()
__END_OF_FILE__
cat > test_pipeline.py << '__END_OF_FILE__'
"""Run with:  python -m unittest -v      (no GPU, no Ollama, no API key needed - models are faked)"""
from __future__ import annotations

import json
import os
import subprocess
import sys
import tempfile
import threading
import unittest
import urllib.error
import urllib.request
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

import finetune_llm
import generate_dataset
import llm_clients
import make_notebook
import pipeline
import shutil
import web_app
from llm_clients import LLMError, OllamaClient, extract_json, pick_default_ollama_model
from retriever import chunk_text

HERE = os.path.dirname(os.path.abspath(__file__))
DOCS = os.path.join(HERE, "sample_docs")


class FakeLLM:
    def __init__(self, label, reply, log):
        self.label, self.reply, self.log = label, reply, log

    def generate_json(self, system, user, retries=1):
        self.log.append((self.label, system, user))
        return self.reply


LLM1_REPLY = {"requirements": [
    {"id": "REQ-001", "statement": "The pool contract shall execute withdrawals below 10,000 USDC equivalent immediately once collateral health checks pass.", "category": "Functional"},
    {"id": "REQ-002", "statement": "The system shall send a push notification to users when the app is updated.", "category": "Usability"}],
    "reasoning": "because"}
LLM2_REPLY = {"recommended_sdlc": "DevSecOps-V-Model Hybrid", "key_factors": ["a", "b"], "justification": "j"}


class RetrievalTests(unittest.TestCase):
    def test_chunks_end_on_paragraph_boundaries(self):
        text = "Para one is short.\n\nPara two is short too.\n\n" + " ".join(["word"] * 300)
        chunks = chunk_text(text, "t.txt", max_words=50, overlap=5)
        self.assertTrue(chunks[0].text.endswith("too."))      # short paragraphs merged, not cut mid-sentence
        self.assertTrue(all(len(c.text.split()) <= 50 for c in chunks))

    def test_retrieval_finds_the_right_document(self):
        r = pipeline.build_index(pipeline.load_documents(DOCS))
        top = lambda d, f: pipeline.retrieve_evidence(r, d, f, 4)[0]["source"]
        self.assertEqual(top("DeFi Lending", "Multi-signature approval for large withdrawals"), "defi_lending_policy.txt")
        self.assertEqual(top("Blockchain KYC / AML", "Sanctions screening of wallet addresses"), "kyc_aml_policy.txt")
        self.assertEqual(top("CBDC", "Offline retail payments"), "cbdc_policy.txt")

    def test_empty_inputs_fail_clearly(self):
        with self.assertRaises(ValueError):
            pipeline.load_documents("", "")
        with self.assertRaises(ValueError):
            pipeline.load_documents("/definitely/not/a/folder")


class FlowTests(unittest.TestCase):
    def test_retrieve_first_then_llm1_then_llm2_sdlc_only(self):
        log = []
        r = pipeline.build_index(pipeline.load_documents(DOCS))
        result = pipeline.run_pipeline(r, "DeFi Lending", "Multi-signature approval for large withdrawals",
                                       FakeLLM("llm1", LLM1_REPLY, log), FakeLLM("llm2", LLM2_REPLY, log))
        self.assertEqual([c[0] for c in log], ["llm1", "llm2"])                 # LLM 2 runs after LLM 1
        _, _, llm1_prompt = log[0]
        self.assertIn("[E1]", llm1_prompt)                                       # LLM 1 already sees retrieved evidence
        self.assertIn(result["evidence"][0]["text"][:40], llm1_prompt)
        _, llm2_system, llm2_prompt = log[1]
        self.assertIn("do not propose or revise requirements", llm2_system)      # LLM 2 is SDLC-only
        self.assertIn("REQ-001", llm2_prompt)                                    # ...and receives the requirements
        self.assertIn("[E1]", llm2_prompt)                                       # ...plus the evidence
        self.assertEqual(result["sdlc"]["recommended_sdlc"], "DevSecOps-V-Model Hybrid")
        self.assertEqual(result["evidence"][0]["source"], "defi_lending_policy.txt")

    def test_grounding_flags_unsupported_requirement(self):
        log = []
        r = pipeline.build_index(pipeline.load_documents(DOCS))
        result = pipeline.run_pipeline(r, "DeFi Lending", "Multi-signature approval for large withdrawals",
                                       FakeLLM("llm1", LLM1_REPLY, log), FakeLLM("llm2", LLM2_REPLY, log))
        supported, unsupported = result["requirements"]
        self.assertFalse(supported["needs_review"])
        self.assertTrue(unsupported["needs_review"])                             # nothing in the evidence about push notifications
        self.assertEqual(result["needs_review_count"], 1)

    def test_no_evidence_flags_everything(self):
        r = pipeline.build_index(pipeline.load_documents("", "Totally unrelated text about gardening and tomatoes."))
        ev = pipeline.retrieve_evidence(r, "DeFi Lending", "withdrawals", 4)
        self.assertEqual(ev, [])
        grounded = pipeline.ground_requirements(r, [{"id": "REQ-001", "statement": "The pool shall do x.", "category": "F"}], ev)
        self.assertTrue(grounded[0]["needs_review"])

    def test_normalisation_copes_with_messy_model_output(self):
        reqs = pipeline.normalize_requirements(["The system shall do A.", {"Requirement": "The system shall do B.", "Type": "Security"}, 5, {}])
        self.assertEqual([r["id"] for r in reqs], ["REQ-001", "REQ-002"])
        self.assertEqual(reqs[1]["category"], "Security")
        self.assertEqual(pipeline.normalize_sdlc({"SDLC": "Agile", "factors": "one"})["key_factors"], ["one"])

    def test_model_with_no_usable_output_raises_clear_error(self):
        r = pipeline.build_index(pipeline.load_documents(DOCS))
        with self.assertRaises(LLMError):
            pipeline.generate_requirements("DeFi Lending", "x", [], FakeLLM("llm1", {"requirements": []}, []))


class ClientTests(unittest.TestCase):
    def test_extract_json_handles_fences_and_chatter(self):
        self.assertEqual(extract_json('```json\n{"a": 1}\n```'), {"a": 1})
        self.assertEqual(extract_json('Sure! Here you go: {"a": 1} Hope that helps'), {"a": 1})
        with self.assertRaises(ValueError):
            extract_json("no json here")

    def test_unreachable_ollama_gives_actionable_error(self):
        with self.assertRaises(LLMError) as ctx:
            OllamaClient("whatever", host="http://127.0.0.1:9", timeout=3).generate_json("s", "u")
        self.assertIn("Cannot reach Ollama", str(ctx.exception))

    def test_default_model_prefers_finetuned_when_installed(self):
        self.assertEqual(pick_default_ollama_model("llm1", ["blockchain-req-llm:latest", "llama3.2:3b"]),
                         {"model": "blockchain-req-llm", "finetuned": True})
        self.assertEqual(pick_default_ollama_model("llm2", ["llama3.2:3b"]), {"model": "qwen2.5:3b", "finetuned": False})
        self.assertEqual(pick_default_ollama_model("llm1", None)["model"], "llama3.2:3b")
        self.assertNotEqual(llm_clients.DEFAULT_OLLAMA_MODELS["llm1"], llm_clients.DEFAULT_OLLAMA_MODELS["llm2"])


class DatasetAndFinetuneTests(unittest.TestCase):
    def test_dataset_is_valid_and_uses_runtime_prompts(self):
        s1, s2, problems = generate_dataset.build_examples(DOCS)
        self.assertEqual(problems, [])
        self.assertGreaterEqual(len(s1), 28)
        self.assertEqual(len(s1), len(s2))
        for line in s1 + s2:
            msgs = json.loads(line)["messages"]
            self.assertEqual([m["role"] for m in msgs], ["system", "user", "assistant"])
            json.loads(msgs[2]["content"])                                       # assistant answer is valid JSON
        user1 = json.loads(s1[0])["messages"][1]["content"]
        self.assertIn("Evidence retrieved from the existing requirement documents", user1)
        self.assertIn(pipeline.STAGE1_SCHEMA, user1)
        labels = {json.loads(json.loads(l)["messages"][2]["content"])["recommended_sdlc"] for l in s2}
        self.assertGreaterEqual(len(labels), 6)                                  # not collapsed to one answer

    def test_training_text_matches_inference_format(self):
        msgs = [{"role": "system", "content": "SYS"}, {"role": "user", "content": "USR"}, {"role": "assistant", "content": "ANS"}]
        for family in ("llama", "qwen"):
            full = finetune_llm.render_example(msgs, family)
            prompt = finetune_llm.render_prompt("SYS", "USR", family)
            self.assertTrue(full.startswith(prompt))                             # training text = inference prompt + answer
            self.assertTrue(full[len(prompt):].startswith("ANS"))
            self.assertIn(finetune_llm.FORMATS[family]["response_part"], full)
            mf = finetune_llm.modelfile_text("model.Q4_K_M.gguf", family)
            self.assertTrue(mf.startswith("FROM ./model.Q4_K_M.gguf"))
            self.assertIn("{{ .System }}", mf)
            self.assertTrue(all(f'PARAMETER stop "{s}"' in mf for s in finetune_llm.FORMATS[family]["stops"]))

    def test_find_gguf_prefers_q4_and_skips_f16(self):
        with tempfile.TemporaryDirectory() as tmp:
            work = os.path.join(tmp, "m_gguf_work")
            os.makedirs(work + "_gguf")
            for n in ("unsloth.F16.gguf", "unsloth.Q4_K_M.gguf"):
                open(os.path.join(work + "_gguf", n), "w").close()
            self.assertTrue(finetune_llm.find_gguf(work).endswith("unsloth.Q4_K_M.gguf"))
            with self.assertRaises(FileNotFoundError):
                finetune_llm.find_gguf(os.path.join(tmp, "nothing"))


class WebApiTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.log = []
        cls._orig = web_app.build_client
        web_app.build_client = lambda provider, model=None, api_key=None, role="llm1": FakeLLM(
            "llm1" if role == "llm1" else "llm2", LLM1_REPLY if role == "llm1" else LLM2_REPLY, cls.log)
        cls.server = ThreadingHTTPServer(("127.0.0.1", 0), web_app.Handler)
        cls.port = cls.server.server_address[1]
        threading.Thread(target=cls.server.serve_forever, daemon=True).start()

    @classmethod
    def tearDownClass(cls):
        cls.server.shutdown()
        cls.server.server_close()
        web_app.build_client = cls._orig

    def _call(self, path, body=None):
        req = urllib.request.Request(f"http://127.0.0.1:{self.port}{path}", method="POST" if body is not None else "GET",
                                     data=json.dumps(body).encode() if body is not None else None,
                                     headers={"Content-Type": "application/json"})
        try:
            with urllib.request.urlopen(req, timeout=10) as r:
                return r.status, json.loads(r.read())
        except urllib.error.HTTPError as e:
            return e.code, json.loads(e.read())

    def test_page_and_config(self):
        with urllib.request.urlopen(f"http://127.0.0.1:{self.port}/", timeout=10) as r:
            html = r.read().decode()
        self.assertIn("Run pipeline", html)
        status, cfg = self._call("/api/config")
        self.assertEqual(status, 200)
        self.assertIn("defaults", cfg)

    def test_three_step_flow_through_http(self):
        base = {"docs": "sample_docs", "domain": "DeFi Lending", "functionality": "Multi-signature approval for large withdrawals", "top_k": 4}
        s, r1 = self._call("/api/retrieve", base)
        self.assertEqual(s, 200)
        self.assertEqual(r1["evidence"][0]["source"], "defi_lending_policy.txt")
        s, r2 = self._call("/api/requirements", {**base, "evidence": r1["evidence"], "provider": "ollama"})
        self.assertEqual(s, 200)
        self.assertEqual(len(r2["requirements"]), 2)
        s, r3 = self._call("/api/sdlc", {"domain": "DeFi Lending", "evidence": r1["evidence"], "requirements": r2["requirements"], "provider": "ollama"})
        self.assertEqual(s, 200)
        self.assertEqual(r3["sdlc"]["recommended_sdlc"], "DevSecOps-V-Model Hybrid")

    def test_bad_input_returns_clean_errors(self):
        s, d = self._call("/api/retrieve", {"docs": "sample_docs", "domain": "", "functionality": "x"})
        self.assertEqual(s, 400)
        self.assertIn("required", d["error"])
        s, d = self._call("/api/retrieve", {"docs": "no_such_folder", "domain": "a", "functionality": "b"})
        self.assertEqual(s, 400)
        self.assertEqual(self._call("/nope", {})[0], 404)


class FakeOllamaHandler(BaseHTTPRequestHandler):
    """Speaks just enough of Ollama's HTTP API to test our real client + CLI code paths."""
    chat_calls = []
    garbage_first = False

    def _send(self, code, payload):
        data = json.dumps(payload).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        self._send(200, {"models": [{"name": "blockchain-req-llm:latest"}, {"name": "qwen2.5:3b"}]})

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        FakeOllamaHandler.chat_calls.append(body)
        if body["model"] == "missing-model":
            return self._send(404, {"error": "model not found"})
        if FakeOllamaHandler.garbage_first and len(FakeOllamaHandler.chat_calls) == 1:
            return self._send(200, {"message": {"content": "Sorry, here are some thoughts..."}})
        reply = LLM2_REPLY if "SDLC" in body["messages"][0]["content"] else LLM1_REPLY
        self._send(200, {"message": {"content": json.dumps(reply)}})

    def log_message(self, *a):
        pass


class OllamaProtocolTests(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.server = ThreadingHTTPServer(("127.0.0.1", 0), FakeOllamaHandler)
        cls.host = f"http://127.0.0.1:{cls.server.server_address[1]}"
        threading.Thread(target=cls.server.serve_forever, daemon=True).start()

    @classmethod
    def tearDownClass(cls):
        cls.server.shutdown()
        cls.server.server_close()

    def setUp(self):
        FakeOllamaHandler.chat_calls.clear()
        FakeOllamaHandler.garbage_first = False

    def test_request_format_and_success(self):
        out = OllamaClient("qwen2.5:3b", host=self.host).generate_json("sys", "usr")
        self.assertEqual(out, LLM1_REPLY)
        sent = FakeOllamaHandler.chat_calls[0]
        self.assertEqual((sent["model"], sent["format"], sent["stream"]), ("qwen2.5:3b", "json", False))
        self.assertEqual([m["role"] for m in sent["messages"]], ["system", "user"])

    def test_invalid_json_is_retried_once(self):
        FakeOllamaHandler.garbage_first = True
        self.assertEqual(OllamaClient("qwen2.5:3b", host=self.host).generate_json("sys", "usr"), LLM1_REPLY)
        self.assertEqual(len(FakeOllamaHandler.chat_calls), 2)
        self.assertIn("not valid JSON", FakeOllamaHandler.chat_calls[1]["messages"][1]["content"])

    def test_missing_model_message_tells_you_what_to_run(self):
        with self.assertRaises(LLMError) as ctx:
            OllamaClient("missing-model", host=self.host).generate_json("s", "u")
        self.assertIn("ollama pull missing-model", str(ctx.exception))

    def test_cli_end_to_end_uses_two_different_models(self):
        env = {**os.environ, "OLLAMA_HOST": self.host}
        run = subprocess.run([sys.executable, os.path.join(HERE, "main.py"), "--domain", "DeFi Lending",
                              "--functionality", "Multi-signature approval for large withdrawals"],
                             capture_output=True, text=True, env=env, timeout=60)
        self.assertEqual(run.returncode, 0, run.stderr)
        self.assertIn("Recommended: DevSecOps-V-Model Hybrid", run.stdout)
        self.assertIn("NEEDS HUMAN REVIEW", run.stdout)
        models = [c["model"] for c in FakeOllamaHandler.chat_calls]
        self.assertEqual(models, ["blockchain-req-llm", "qwen2.5:3b"])      # fine-tuned LLM 1 auto-detected; LLM 2 falls back to base


class NotebookTests(unittest.TestCase):
    def setUp(self):
        self.nb = make_notebook.build()
        self.bash_cells = [c for c in self.nb["cells"] if c["cell_type"] == "code" and "".join(c["source"]).startswith("%%bash")]

    def test_notebook_structure(self):
        json.dumps(self.nb)                                            # serialisable
        own = [c for c in self.nb["cells"] if c["cell_type"] == "code" and c not in self.bash_cells]
        code = "\n".join("".join(c["source"]) for c in own)           # notebook's own cells (not the embedded project files)
        for needle in ("nvidia-smi", "pip install -q unsloth", "generate_dataset.py",
                       "finetune_llm.py --stage 1", "finetune_llm.py --stage 2", "drive.mount"):
            self.assertIn(needle, code)
        self.assertNotIn("zipfile.ZipFile(list(uploaded)", code)       # the fragile zip-upload step is gone
        self.assertEqual(len(self.bash_cells), 2)

    @unittest.skipUnless(shutil.which("bash"), "needs bash (Colab has it)")
    def test_bash_cells_recreate_every_project_file_exactly(self):
        """Run the notebook's file-creation cells the way Colab does (script fed to bash on stdin)."""
        with tempfile.TemporaryDirectory() as tmp:
            for cell in self.bash_cells:
                script = "".join(cell["source"]).split("\n", 1)[1]    # drop the %%bash magic line
                run = subprocess.run(["bash"], input=script, text=True, cwd=tmp, capture_output=True)
                self.assertEqual(run.returncode, 0, run.stderr)
            wanted = make_notebook.TRAIN_FILES + make_notebook.EXTRA_FILES + make_notebook._doc_files()
            for rel in wanted:
                original = open(os.path.join(HERE, rel), encoding="utf-8").read().replace("\r\n", "\n").rstrip("\n") + "\n"
                with open(os.path.join(tmp, rel), encoding="utf-8") as f:
                    self.assertEqual(f.read(), original, f"{rel} was not recreated exactly")
            # the recreated project must actually work: build the datasets inside the fresh folder
            run = subprocess.run([sys.executable, "generate_dataset.py"], cwd=tmp, capture_output=True, text=True)
            self.assertEqual(run.returncode, 0, run.stderr)
            self.assertTrue(os.path.exists(os.path.join(tmp, "train_stage1.jsonl")))


if __name__ == "__main__":
    unittest.main(verbosity=2)
__END_OF_FILE__
cat > requirements.txt << '__END_OF_FILE__'
# Needed to run the website / CLI (works fully offline with Ollama)
scikit-learn>=1.3
numpy>=1.24

# Optional - only if you want to use Gemini (cloud) instead of / alongside Ollama:
# google-genai>=1.0
__END_OF_FILE__
cat > README.md << '__END_OF_FILE__'
# Blockchain Finance: Requirements and SDLC Pipeline

A retrieval-grounded, two-model pipeline for blockchain-based financial software projects. Given a domain and a
functionality, the system retrieves evidence from a set of existing requirement documents, generates grounded
requirements with an explanation, and then recommends the most suitable Software Development Life Cycle (SDLC)
model with a justification.

Both language models run locally on [Ollama](https://ollama.com), so the system works offline and needs no API key.
Each model can optionally be replaced by a fine-tuned version produced with the included training workflow.

## Overview

```
 input: domain + functionality
    |
    v
 1. RAG retrieval   Finds the most relevant chunks of the existing requirement documents (TF-IDF).
    |  evidence
    v
 2. LLM 1           Llama 3.2 3B (or the fine-tuned blockchain-req-llm).
    |               Writes requirements and reasoning, grounded in the evidence.
    |               A grounding check then scores every requirement against the evidence and
    |               flags weakly supported ones for human review.
    v
 3. LLM 2           Qwen 2.5 3B (or the fine-tuned blockchain-sdlc-llm).
    |               Identifies the SDLC model only: recommendation, key factors, justification.
    v
 final output
```

**Supported SDLC outcomes:** Waterfall, V-Model, Spiral, Agile, DevSecOps, Agile-V-Model Hybrid and
DevSecOps-V-Model Hybrid.

**Sample domains:** DeFi lending, cross-border payments, KYC/AML, stablecoin issuance, crypto custody, tokenized
trade finance and CBDC. Other domains work as long as relevant documents are supplied.

## Requirements

- Python 3.9 or newer
- [Ollama](https://ollama.com) (running locally)
- About 2 GB of disk space per model, and 8 GB of RAM or more recommended
- Optional: a Google account for fine-tuning on free Colab GPUs; a Gemini API key to use Gemini instead of Ollama

## Quick start

### 1. Install the Python dependencies

```
pip install -r requirements.txt
```

### 2. Install Ollama and the base models

Install Ollama from https://ollama.com and make sure it is running (open the app, or run `ollama serve`). Opening
http://localhost:11434 in a browser should show "Ollama is running". Then download the two base models once:

```
ollama pull llama3.2:3b
ollama pull qwen2.5:3b
```

### 3. Start the web application

```
python web_app.py
```

The browser opens at http://localhost:8000 (set the `PORT` environment variable to use another port). Enter a
domain and a functionality, then select **Run pipeline**. The status box at the top shows whether Ollama is reachable
and which models will be used. The three steps run one after another and each result appears as soon as it is ready.
On a laptop CPU, each LLM step typically takes from one to a few minutes; a GPU is considerably faster.

Example input:

| Field | Value |
|---|---|
| Domain | `DeFi Lending` |
| Functionality | `Governance timelock before changes to lending pool parameters take effect` |

### Command-line usage

```
python main.py --domain "DeFi Lending" --functionality "Multi-signature approval for large withdrawals"
python main.py --domain "CBDC" --functionality "Offline retail payments" --json-out result.json
```

| Option | Default | Description |
|---|---|---|
| `--docs` | `sample_docs` | Folder of `.txt` / `.md` requirement documents |
| `--llm1-provider`, `--llm1-model` | `ollama`, auto | Model for requirement generation |
| `--llm2-provider`, `--llm2-model` | `ollama`, auto | Model for SDLC identification |
| `--top-k` | `4` | Number of evidence chunks retrieved |
| `--min-grounding` | `0.10` | Support score below which a requirement is flagged for review |
| `--json-out` | none | Save the full result to a JSON file |

When no model is specified, the fine-tuned model is used if it is installed in Ollama, and the base model otherwise.

### Using custom documents

Place `.txt` or `.md` files in a folder and enter its path in the "Document folder" field (or pass `--docs`).
Additional text can be pasted into the optional box on the web page. The files in `sample_docs/` are illustrative
policy notes created for this project; they are not real regulations.

### Using Gemini (optional)

```
pip install google-genai
```

Set the `GEMINI_API_KEY` environment variable (or paste the key into the web page) and choose "Gemini" as the
provider for either step. The default model is `gemini-3.5-flash-lite`.

## Fine-tuning the models (optional)

The pipeline runs fully on the base models. Fine-tuning is useful when a project requires fine-tuned models, or
when the base models produce malformed or weak output.

**What it changes.** Fine-tuning teaches the small models the expected output style: valid JSON, one-sentence
"shall" requirements, use of the supplied evidence, and a justified choice among the seven SDLC outcomes. It does
not reliably add domain knowledge; that comes from the retrieved documents.

### Training data

`seed_data.py` contains 28 synthetic scenarios across seven domains. They are hand-authored examples written for
this project and are not drawn from real organisational or regulatory documents, so they should be reviewed and
edited before being relied upon. Each scenario yields one training example for each model:

- **LLM 1:** domain, functionality and retrieved evidence, producing requirements and reasoning.
- **LLM 2:** the same requirements and evidence, producing the SDLC model, key factors and justification.

The SDLC labels are deliberately varied (Waterfall 4, V-Model 5, Spiral 4, Agile 3, DevSecOps 3, hybrids 9) so that
the model does not default to a single answer. The evidence in every training prompt is produced by running the real
retriever over `sample_docs/`, and the prompts are the ones defined in `pipeline.py`, so training and inference use
identical formats. The generated files are `train_stage1.jsonl` and `train_stage2.jsonl`.

A set of 28 examples is small. It improves format and style but does not give deep expertise, and some memorisation
of the examples is likely. Adding scenarios to `seed_data.py` (100 or more is recommended, ideally based on real
documents) and regenerating the data improves results. Whenever `sample_docs/` or a prompt in `pipeline.py`
changes, run `python generate_dataset.py` again and retrain.

| Model | Base model | Training data | Name in Ollama |
|---|---|---|---|
| LLM 1 (requirements) | Llama-3.2-3B-Instruct | `train_stage1.jsonl` | `blockchain-req-llm` |
| LLM 2 (SDLC only) | Qwen2.5-3B-Instruct | `train_stage2.jsonl` | `blockchain-sdlc-llm` |

Training uses QLoRA through [Unsloth](https://docs.unsloth.ai), with a LoRA rank of 16, three epochs and a learning
rate of 2e-4 by default. Loss is computed on the assistant's answer only (when Unsloth's `train_on_responses_only` is available). The models are exported as 4-bit GGUF
files together with an Ollama `Modelfile`.

### Procedure (Google Colab)

`finetune_colab.ipynb` is self-contained: it contains all the project code and creates the required files inside
Colab, so nothing needs to be zipped or uploaded.

1. Open https://colab.research.google.com, choose **File > Upload notebook**, and select `finetune_colab.ipynb`.
2. Choose **Runtime > Change runtime type > T4 GPU** and save.
3. Run the cells in order, waiting for each to finish: check the GPU, create the project files, verify the files,
   install Unsloth, build the training data, fine-tune LLM 1, fine-tune LLM 2.
4. Save the models before the session ends, because Colab deletes its files afterwards. Option A copies them to
   Google Drive; option B downloads one zip per model. Do not restart or disconnect the runtime until the models
   are saved.
5. On the local machine, create a folder named `exports` inside the project folder, so that the layout is:
   ```
   exports/
     blockchain-req-llm/
       Modelfile
       model.Q4_K_M.gguf
     blockchain-sdlc-llm/
       Modelfile
       model.Q4_K_M.gguf
   ```
6. With Ollama running, register the models:
   ```
   python install_models.py
   ollama list
   ```
   Both `blockchain-` models should be listed.
7. Restart `python web_app.py`. The status box reports that the fine-tuned models are in use. No configuration is
   needed, because installed fine-tuned models are selected automatically.

The exported models are about 2 GB each and are excluded from version control by `.gitignore`. They can be
regenerated at any time with the notebook.

After editing `seed_data.py` or any other file, run `python make_notebook.py` to rebuild `finetune_colab.ipynb`
with the current files embedded, then upload the new notebook.

### Using a local GPU instead of Colab

On a Linux or WSL machine with an NVIDIA GPU:

```
pip install unsloth
python generate_dataset.py
python finetune_llm.py --stage 1
python finetune_llm.py --stage 2
python install_models.py
```

### Evaluating the result

Write several functionalities that do not appear in `seed_data.py`, run each with the base models and with the
fine-tuned models (the model name can be typed into the web page), and compare the validity of the output, the
number of requirements flagged for review, and the quality of the SDLC choice and justification.

### Troubleshooting

| Symptom | Resolution |
|---|---|
| Google Drive reports `mount failed` | Allow pop-ups, choose the same Google account Colab uses, accept every screen and run the cell again; or use the direct-download option. |
| Notebook step 3 reports `Missing: ...` | Step 2 did not complete. Run it again and read its output. |
| Unsloth installation conflicts or API errors | Follow the current installation notes at https://docs.unsloth.ai. `finetune_llm.py` already handles the different `SFTTrainer` signatures used by recent TRL versions. |
| Out of disk space on Colab | Delete the `*_gguf_work` and `*_checkpoints` folders of the first stage before training the second (the notebook does this). |
| `No quantised .gguf found` | Look inside `exports/<name>_gguf_work*` and copy the Q4_K_M file next to the Modelfile as `model.Q4_K_M.gguf`. |
| Model returns invalid JSON | Reduce `--epochs` (over-training) or add more varied training examples. |
| Status box says a model is not installed | Run `ollama list`. Use `ollama pull <name>` for base models or `python install_models.py` for fine-tuned ones. |

## Testing

```
python -m unittest -v
```

The suite contains 23 tests and uses fake models, so no GPU, Ollama installation or API key is required. It covers
paragraph-aware chunking, retrieval of the correct document, the pipeline order (retrieval before LLM 1, and LLM 2
restricted to the SDLC decision and run after LLM 1), grounding flags, handling of malformed model output, the
Ollama HTTP requests (including the JSON retry and the missing-model message), the command-line interface, the
generated datasets, consistency between the training text and the Ollama prompt format, the three-step web API, and
the notebook's recreation of every project file.

## Design notes

- **Retrieval before generation.** Evidence is retrieved first and supplied to LLM 1, so requirements are grounded
  in the existing documents. LLM 2 receives the same evidence together with the requirements.
- **Two different models.** Requirement writing and SDLC selection are separate tasks with separate models, and
  LLM 2 is explicitly restricted to the SDLC decision.
- **Deterministic grounding check.** Each requirement is compared with the retrieved evidence using TF-IDF cosine
  similarity. This does not depend on the LLM, and weak support triggers a "needs human review" flag.
- **Shared prompts.** The prompt builders in `pipeline.py` are used by the pipeline and by the dataset generator.
- **Robust output handling.** Replies are requested as JSON, parsed tolerantly, retried once on invalid JSON, and
  normalised before use. Model output is escaped before it is displayed in the web page.

## Limitations

- TF-IDF retrieval is keyword-based and can miss paraphrased content. A neural embedding retriever can replace it
  by providing the same `index`, `retrieve_scored` and `best_support` methods.
- The grounding score measures lexical similarity, not correctness. Requirements that are not flagged still need
  human review.
- Models of three billion parameters give modest text quality. Larger Ollama models or Gemini produce better results
  at the cost of speed or local execution.
- The sample documents and training scenarios are synthetic and illustrative.
- Unsloth and TRL change frequently. If the training notebook fails after a library update, the version-specific
  calls in `finetune_llm.py` may need adjusting.

## Project structure

| File | Purpose |
|---|---|
| `web_app.py` | Web application (`python web_app.py`) |
| `main.py` | Command-line interface |
| `pipeline.py` | The three pipeline steps and shared prompts |
| `retriever.py` | Chunking, TF-IDF retrieval and grounding score |
| `llm_clients.py` | Ollama and Gemini clients, JSON repair, fine-tuned model detection |
| `seed_data.py` | 28 synthetic training scenarios |
| `generate_dataset.py` | Builds `train_stage1.jsonl` and `train_stage2.jsonl` |
| `finetune_llm.py` | Fine-tunes one model with Unsloth QLoRA and exports GGUF and Modelfile |
| `finetune_colab.ipynb` | Self-contained Colab notebook for fine-tuning |
| `make_notebook.py` | Rebuilds the notebook with the current project files embedded |
| `install_models.py` | Registers the exported models in Ollama |
| `test_pipeline.py` | Automated tests |
| `sample_docs/` | Seven illustrative requirement and policy documents (the retrieval corpus) |
| `.gitignore` | Excludes exported models, caches and local files from version control |
__END_OF_FILE__
echo "Created $(ls | wc -l) items:"; ls

In [ ]:
!zip -qr blockchain_rag_project_code.zip *.py *.md requirements.txt sample_docs train_stage1.jsonl train_stage2.jsonl
from google.colab import files
files.download("blockchain_rag_project_code.zip")

### 11. On your own computer
1. Put the models in a folder named `exports` inside your project folder, so you have `exports/blockchain-req-llm/` and
   `exports/blockchain-sdlc-llm/` (each with a `Modelfile` and `model.Q4_K_M.gguf`). Option A: download the `blockchain_llms`
   folder from Google Drive and rename it `exports`. Option B: unzip both downloaded zips into `exports`.
2. Make sure the Ollama app is running, then in the project folder run: `python install_models.py`
3. Start the website: `python web_app.py` - the status box will say it is using your fine-tuned models.